# Edge-Efficient Computational Pathology — Optimization Study
### NB3 → NB6: Freeze, Baselines, Seeds, Optimize, Benchmark, Analyse
**Strategy:** `mobilevit_s` (topology-free) → ONNX → FP16 + second-generation mixed-precision INT8 PTQ → T4 batch-1 benchmark → Pareto analysis  
**Hardware:** Kaggle NVIDIA T4 GPU (controlled proxy for GPU-based edge-oriented inference)  
**Training is used only for the explicitly configured architecture-baseline study; the MobileViT deployment checkpoint remains frozen. No QAT. No pruning.**

## § 0 — Environment & Dependencies

In [ ]:
# Install profiling and ONNX stack
# IMPORTANT: install ONLY onnxruntime-gpu, not both onnxruntime + onnxruntime-gpu.
# They share the same import namespace ('onnxruntime'); installing both lets one
# silently shadow the other and CUDAExecutionProvider can disappear with no error.
# onnxruntime-gpu already includes CPUExecutionProvider, so it alone covers both
# GPU (FP32/FP16) and CPU (INT8) inference in this notebook.
!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>/dev/null
# BUGFIX: unpinned `onnxruntime-gpu` now resolves to >=1.27, which PyPI builds
# against CUDA 13.0 by default (needs libcublasLt.so.13 + cuDNN 9.x for CUDA 13).
# Kaggle's GPU images still ship a CUDA 12.x runtime, so that wheel's CUDA EP
# fails to load (`libcublasLt.so.13: cannot open shared object file`). ORT's
# `InferenceSession` swallows that failure and silently falls back to CPU, so
# accuracy eval (cell above) still "worked" — just secretly on CPU. IOBinding's
# `ortvalue_from_numpy(..., 'cuda', 0)` has no such fallback and raises instead,
# which is where the run actually died. Pin to the last CUDA-12.8-default line
# (<1.27) so the installed wheel matches Kaggle's CUDA 12.x runtime.
!pip install -q onnx onnxscript "onnxruntime-gpu<1.27" onnxconverter-common fvcore scipy


## § 1 — Imports & Device

In [ ]:
import os, json, time, glob, random, warnings, copy
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F_
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, roc_curve
from tqdm.auto import tqdm
import timm

import onnx
import onnxruntime as ort
from onnxconverter_common import float16
from fvcore.nn import FlopCountAnalysis, parameter_count

# ── Device ────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## § 2 — CONFIG & Paths

In [ ]:
# ── Kaggle input dataset slugs ─────────────────────────────────────────────
# Adjust these dataset slugs to match what you have attached to this Kaggle notebook.
# NB1_DIR  → your topology_checkpoint output (from NB1)
# PREV_DIR → the mobilevit_life_results output (from your Stage-2 notebook)
# PCAM_DIR → the PatchCamelyon Kaggle dataset root

NB1_DIR  = '/kaggle/input/datasets/salon123/final-final-dataset'
PREV_DIR = '/kaggle/input/datasets/salon123/final-final-dataset'   # contains *_deploy_backbone.pt
PCAM_DIR = '/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon'
OUT_DIR  = '/kaggle/working/edge_opt_results'
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    'BACKBONE':          'mobilevit_s',
    'IMG_SIZE':          96,
    'BATCH_SIZE':        256,       # eval batch
    'NUM_WORKERS':       2,
    'VAL_SUBSET':        5000,
    'TEST_SUBSET':       5000,
    'STAGE2_SEEDS':      [0, 1, 2],
    # CODS-style baseline/seed study. Baselines are trained with the same
    # PCam input pipeline and evaluated on the SAME frozen val/test subsets.
    # 2024-01+ note: the original 3 baselines (ResNet18, MobileNetV3, EfficientNet-B0)
    # all predate MobileViT (2021) or are same-generation CNNs. They don't answer the
    # key reviewer question -- "is MobileViT actually competitive against *newer*
    # mobile/edge architectures?" -- so we add four post-2021 mobile-efficient hybrid
    # CNN-transformer models, each sanity-checked to run at IMG_SIZE=96 with
    # num_classes=2 before being added here:
    #   - edgenext_x_small   : EdgeNeXt (2022)      -- hybrid conv+attention, mobile-optimized
    #   - mobilevitv2_050    : MobileViTv2 (2022)    -- direct successor to this paper's MobileViT-S backbone
    #   - fastvit_t8         : FastViT (2023, Apple) -- structural reparam, mobile-optimized
    #   - repvit_m0_9        : RepViT (2023/24)      -- reparameterized CNN designed to beat mobile ViTs
    # (efficientformerv2_s0 was tried and rejected: its windowed attention reshape
    # assumes resolutions that don't evenly divide at 96x96 and throws a runtime
    # shape error, so it's not included.)
    'BASELINE_MODELS':   ['resnet18', 'mobilenetv3_small_100', 'efficientnet_b0',
                           'edgenext_x_small', 'mobilevitv2_050', 'fastvit_t8', 'repvit_m0_9'],
    'BASELINE_SEEDS':    [0, 1, 2],
    'BASELINE_EPOCHS':   8,
    'BASELINE_LR':       3e-4,
    'BASELINE_WEIGHT_DECAY': 1e-4,
    # 50k is a practical Kaggle run. Set to None for the full PCam training split
    # for the final archival-paper run.
    'BASELINE_TRAIN_SUBSET': 50000,
    'BASELINE_BATCH_SIZE': 128,
    'BASELINE_PRETRAINED': True,
    'BASELINE_NUM_WORKERS': 2,
    'WARMUP_ITERS':      50,        # benchmark warm-up iterations
    'BENCH_ITERS':       1000,      # benchmark measurement iterations
    'CALIB_IMAGES':      1000,      # INT8 PTQ calibration images (bumped 100->500->1000:
                                # too few images gives noisy activation ranges,
                                # which hurts attention-heavy nets especially;
                                # now drawn class-balanced, see calib_idx below)
    'OUT_DIR':           OUT_DIR,
}

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(0)
print(json.dumps(CONFIG, indent=2))

## § 3 — Data Setup (PCam Val + Test Loaders)

In [ ]:
# ── Discover PCam H5 files via glob (matches the existing notebooks' pattern) ──
def find_pcam_files(root):
    root = Path(root)
    all_h5 = list(root.rglob('*.h5'))
    image_names = {
        'train_x': 'training_split.h5',
        'valid_x': 'validation_split.h5',
        'test_x':  'test_split.h5',
    }
    label_names = {
        'train_y': 'camelyonpatch_level_2_split_train_y.h5',
        'valid_y': 'camelyonpatch_level_2_split_valid_y.h5',
        'test_y':  'camelyonpatch_level_2_split_test_y.h5',
    }
    found = {}
    for key, filename in {**image_names, **label_names}.items():
        matches = [f for f in all_h5 if f.name == filename]
        if matches:
            found[key] = str(matches[0])
    return found

pcam_files = find_pcam_files(PCAM_DIR)
missing = [k for k in ['valid_x', 'valid_y', 'test_x', 'test_y'] if k not in pcam_files]
if missing:
    raise FileNotFoundError(f'Missing PCam files: {missing}. Check PCAM_DIR.')
print('PCam files found:')
for k, v in pcam_files.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Pure-image dataset for the deploy backbone (no topology needed) ─────────
class PCamImageDataset(Dataset):
    """Minimal PCam dataset: returns only (img_tensor, label).
    No topology maps required — the deploy backbone is topology-free."""
    def __init__(self, h5_image_path, indices, labels):
        self.h5_path = h5_image_path
        self.indices = np.asarray(indices)
        self.labels  = np.asarray(labels)
        self._h5 = None

    def _images(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, 'r')
            self._key = list(self._h5.keys())[0]
        return self._h5[self._key]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        img = np.array(self._images()[idx]).astype(np.float32) / 255.0
        img_t = torch.from_numpy(img.transpose(2, 0, 1).copy()).float()
        label = int(self.labels[i])
        return img_t, label


def load_labels(split, n):
    with h5py.File(pcam_files[f'{split}_y'], 'r') as f:
        y_key = list(f.keys())[0]
        return np.array(f[y_key]).reshape(-1)[:n]


rng = np.random.default_rng(0)

# Read full dataset sizes from the H5 files
with h5py.File(pcam_files['valid_x'], 'r') as f:
    n_valid_full = list(f.values())[0].shape[0]
with h5py.File(pcam_files['test_x'], 'r') as f:
    n_test_full = list(f.values())[0].shape[0]

y_valid_full = load_labels('valid', n_valid_full)
y_test_full  = load_labels('test',  n_test_full)

val_idx         = rng.choice(n_valid_full, min(CONFIG['VAL_SUBSET'],  n_valid_full),  replace=False)
test_idx        = rng.choice(n_test_full,  min(CONFIG['TEST_SUBSET'], n_test_full),   replace=False)
val_labels_sub  = y_valid_full[val_idx]
test_labels_sub = y_test_full[test_idx]

# ── Stratified calibration subset (both classes, larger N) ─────────────────
# Calibration coverage matters more than calibration/eval overlap: draw a
# class-balanced sample directly from the full validation pool rather than
# reusing a random slice of val_idx, so INT8 activation ranges are fit on a
# guaranteed mix of tumor/normal patches instead of whatever ratio a random
# draw happens to land on.
def stratified_indices(labels_full, n_total, rng):
    classes = np.unique(labels_full)
    n_per_class = n_total // len(classes)
    idx_parts = []
    for c in classes:
        pool = np.flatnonzero(labels_full == c)
        take = min(n_per_class, len(pool))
        idx_parts.append(rng.choice(pool, take, replace=False))
    idx = np.concatenate(idx_parts)
    rng.shuffle(idx)
    return idx

calib_idx = stratified_indices(y_valid_full, CONFIG['CALIB_IMAGES'], rng)
print(f'Calibration subset: {len(calib_idx)} images '
      f'({(y_valid_full[calib_idx] == 1).sum()} tumor / '
      f'{(y_valid_full[calib_idx] == 0).sum()} normal)')

val_ds   = PCamImageDataset(pcam_files['valid_x'], val_idx,  val_labels_sub)
test_ds  = PCamImageDataset(pcam_files['test_x'],  test_idx, test_labels_sub)

val_loader  = DataLoader(val_ds,  batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=CONFIG['NUM_WORKERS'])
test_loader = DataLoader(test_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=CONFIG['NUM_WORKERS'])

print(f'Val subset:  {len(val_idx)} images')
print(f'Test subset: {len(test_idx)} images')

---
## NB3 — Freeze & Golden Baseline
Load all three Stage-2 deploy backbones via `glob`, select the best seed by val AUC, verify identity, profile, and save the golden model.

In [ ]:
# ── Discover all Stage-2 deploy backbones via glob ──────────────────────────
deploy_pattern = os.path.join(PREV_DIR, '*_deploy_backbone.pt')
deploy_files   = sorted(glob.glob(deploy_pattern))

if not deploy_files:
    raise FileNotFoundError(
        f'No deploy backbones found at pattern: {deploy_pattern}\n'
        f'Attach your Stage-2 output dataset and update PREV_DIR.'
    )

print(f'Found {len(deploy_files)} deploy backbone(s):')
for f in deploy_files:
    print(f'  {os.path.basename(f)}')

In [ ]:
# ── Load per-seed Stage-2 metrics and select best seed by val AUC ───────────
metrics_pattern = os.path.join(PREV_DIR, 'mobilevit_life_consistency_stage2__seed*_metrics.json')
metrics_files   = sorted(glob.glob(metrics_pattern))

seed_records = []
for mf in metrics_files:
    with open(mf) as f:
        m = json.load(f)
    seed = m.get('seed', int(Path(mf).stem.split('seed')[1].split('_')[0]))
    val_auc  = m.get('standard_val',  {}).get('auc', 0.0)
    test_auc = m.get('standard_test', {}).get('auc', 0.0)
    deploy_pt = os.path.join(
        PREV_DIR, f'mobilevit_life_consistency_stage2__seed{seed}_deploy_backbone.pt'
    )
    seed_records.append({
        'seed':       seed,
        'val_auc':    val_auc,
        'test_auc':   test_auc,
        'deploy_pt':  deploy_pt,
        'exists':     os.path.exists(deploy_pt),
    })

# Also include any found files not in the metrics (safety net)
found_seeds = {r['seed'] for r in seed_records}
for df in deploy_files:
    try:
        s = int(Path(df).stem.split('seed')[1].split('_')[0])
    except Exception:
        continue
    if s not in found_seeds:
        seed_records.append({'seed': s, 'val_auc': 0.0, 'test_auc': 0.0, 'deploy_pt': df, 'exists': True})

seed_records = [r for r in seed_records if r['exists']]

df_seeds = pd.DataFrame(seed_records).sort_values('val_auc', ascending=False)
print(df_seeds[['seed','val_auc','test_auc','deploy_pt']].to_string(index=False))

best = df_seeds.iloc[0]
GOLDEN_PT = best['deploy_pt']
BEST_SEED = int(best['seed'])
print(f'\n✓ Best seed: {BEST_SEED}  |  Val AUC: {best["val_auc"]:.4f}  |  Weights: {os.path.basename(GOLDEN_PT)}')

In [ ]:
# ── Load the golden deploy backbone ─────────────────────────────────────────
# The deploy backbone is a pure mobilevit_s with num_classes=2.
# It was saved by extract_deploy_model() in the Stage-2 notebook as a plain state_dict.
deploy_model = timm.create_model(CONFIG['BACKBONE'], pretrained=False, num_classes=2)
state = torch.load(GOLDEN_PT, map_location='cpu')
deploy_model.load_state_dict(state, strict=True)
deploy_model = deploy_model.to(DEVICE).eval()

print(f'✓ Loaded deploy backbone from seed {BEST_SEED}: {CONFIG["BACKBONE"]}')
print(f'  Checkpoint: {os.path.basename(GOLDEN_PT)}')

In [ ]:
# ── Sanity-check: forward pass on a real batch ───────────────────────────────
# Ensures the model outputs valid logits before we do any conversion.
sample_imgs, sample_labels = next(iter(val_loader))
sample_imgs = sample_imgs[:4].to(DEVICE)

with torch.no_grad():
    logits = deploy_model(sample_imgs)
    probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

assert logits.shape == (4, 2), f'Unexpected output shape: {logits.shape}'
assert not np.any(np.isnan(probs)), 'NaN detected in probabilities!'
print(f'✓ Forward pass OK.  Sample probs (class 1): {probs}')

In [ ]:
# ── Profile FP32 baseline metrics ────────────────────────────────────────────
dummy = torch.randn(1, 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']).to(DEVICE)
flops_analysis = FlopCountAnalysis(deploy_model, dummy)
flops_analysis.unsupported_ops_warnings(False)
flops_analysis.uncalled_modules_warnings(False)
total_macs    = flops_analysis.total()
total_params  = parameter_count(deploy_model)['']

# File size
fp32_size_mb  = os.path.getsize(GOLDEN_PT) / (1024 ** 2)

print(f'Parameters:     {total_params:,}')
print(f'MACs (FLOPs/2): {total_macs / 1e6:.1f} M')
print(f'FP32 Size:      {fp32_size_mb:.2f} MB')

In [ ]:
# ── Evaluate FP32 AUC and F1 on val and test subsets ─────────────────────────
@torch.no_grad()
def evaluate_pytorch(model, loader, desc='Evaluating'):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs)
    f1  = f1_score(all_labels, (all_probs > 0.5).astype(int))
    acc = accuracy_score(all_labels, (all_probs > 0.5).astype(int))
    return {'auc': auc, 'f1': f1, 'acc': acc, 'probs': all_probs, 'labels': all_labels}

fp32_val  = evaluate_pytorch(deploy_model, val_loader,  desc='FP32 Val')
fp32_test = evaluate_pytorch(deploy_model, test_loader, desc='FP32 Test')

print(f'FP32  Val  AUC={fp32_val["auc"]:.4f}  F1={fp32_val["f1"]:.4f}  Acc={fp32_val["acc"]:.4f}')
print(f'FP32  Test AUC={fp32_test["auc"]:.4f}  F1={fp32_test["f1"]:.4f}  Acc={fp32_test["acc"]:.4f}')

In [ ]:
# ── Save golden metadata ─────────────────────────────────────────────────────
GOLDEN_DEST = os.path.join(OUT_DIR, 'mobilevit_deploy_golden.pt')
torch.save(deploy_model.state_dict(), GOLDEN_DEST)

golden_meta = {
    'architecture': CONFIG['BACKBONE'],
    'input_size':   [3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']],
    'num_classes':  2,
    'best_seed':    BEST_SEED,
    'source_checkpoint': os.path.basename(GOLDEN_PT),
    'params':       total_params,
    'macs':         total_macs,
    'fp32_size_mb': fp32_size_mb,
    'fp32_val':     {k: float(v) for k, v in fp32_val.items() if k not in ('probs','labels')},
    'fp32_test':    {k: float(v) for k, v in fp32_test.items() if k not in ('probs','labels')},
}

with open(os.path.join(OUT_DIR, 'golden_metadata.json'), 'w') as f:
    json.dump(golden_meta, f, indent=2)

print(f'✓ Golden model saved to:    {GOLDEN_DEST}')
print(f'✓ Metadata saved to:        {OUT_DIR}/golden_metadata.json')
print(json.dumps({k: v for k,v in golden_meta.items() if k not in ('fp32_val','fp32_test')}, indent=2))

## Statistical Utilities — Bootstrap Confidence Intervals

Adds a 95% percentile bootstrap CI for test-set AUC, computed directly from each model's saved prediction probabilities. This is a **different, complementary** source of uncertainty from the cross-seed mean ± SD computed elsewhere in this notebook:

- **Cross-seed SD** (already in the notebook) captures variance from different training runs (init/seed/optimization noise).
- **Bootstrap CI** (added here) captures sampling variance from evaluating on one finite (5,000-image) test subset.

A reviewer can reasonably ask for either, so both are reported.

In [ ]:
# ── Bootstrap CI helper for AUC ─────────────────────────────────────────────
# Percentile bootstrap: resample (label, prob) pairs with replacement B times,
# recompute AUC each time, and report the 2.5th/97.5th percentiles as a 95% CI.
def bootstrap_auc_ci(labels, probs, n_boot=1000, alpha=0.05, seed=0):
    """Percentile bootstrap 95% CI (default) for ROC-AUC.

    Returns (auc_point, ci_low, ci_high). If too many bootstrap resamples are
    degenerate (only one class present -- possible on small/imbalanced
    subsets), the interval is reported as NaN rather than silently computed
    from too few valid resamples.
    """
    labels = np.asarray(labels)
    probs  = np.asarray(probs)
    n = len(labels)
    rng = np.random.default_rng(seed)
    point_auc = roc_auc_score(labels, probs)

    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yl, yp = labels[idx], probs[idx]
        if len(np.unique(yl)) < 2:
            continue
        boot_aucs.append(roc_auc_score(yl, yp))

    if len(boot_aucs) < max(50, n_boot // 10):
        return float(point_auc), float('nan'), float('nan')

    boot_aucs = np.asarray(boot_aucs)
    lo = float(np.percentile(boot_aucs, 100 * (alpha / 2)))
    hi = float(np.percentile(boot_aucs, 100 * (1 - alpha / 2)))
    return float(point_auc), lo, hi


print('\u2713 bootstrap_auc_ci() defined \u2014 95% percentile bootstrap CI for AUC, n_boot=1000 default')


## CODS 2026 — Multi-Seed + Multi-Architecture Baseline Study

This section strengthens the experimental design beyond the single best MobileViT seed.
It reports **all configured MobileViT seeds** and trains/evaluates three compact CNN baselines
(**ResNet-18, MobileNetV3-Small, EfficientNet-B0**) using the same PCam task, frozen validation/test
subsets, input resolution, and evaluation metrics.

**Selection rule:** validation AUC may be used to select a checkpoint *within a seed/model* for
deployment analysis, but the paper-facing table reports the test AUC for every seed and
mean ± standard deviation across seeds. The test set is never used for model selection.

**Important:** the default `BASELINE_TRAIN_SUBSET=50000` is a practical screening setting.
For a final archival run, set it to `None` and report the exact epoch count/early-stopping rule.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CODS BASELINE DATA + REPRODUCIBLE SEED STUDY
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 80)
print("CODS BASELINE + MULTI-SEED STUDY")
print("=" * 80)
print("Baselines:", CONFIG['BASELINE_MODELS'])
print("Seeds    :", CONFIG['BASELINE_SEEDS'])
print("Epochs   :", CONFIG['BASELINE_EPOCHS'])
print("Train N  :", CONFIG['BASELINE_TRAIN_SUBSET'] or 'FULL')

# Read the full PCam training labels.
with h5py.File(pcam_files['train_x'], 'r') as f:
    n_train_full = list(f.values())[0].shape[0]
y_train_full = load_labels('train', n_train_full)

# One fixed, stratified training subset shared by every architecture/seed.
# This isolates seed/architecture effects from different sampled examples.
train_rng = np.random.default_rng(2026)
if CONFIG['BASELINE_TRAIN_SUBSET'] is None:
    train_idx_baseline = np.arange(n_train_full, dtype=np.int64)
else:
    train_idx_baseline = stratified_indices(
        y_train_full,
        min(CONFIG['BASELINE_TRAIN_SUBSET'], n_train_full),
        train_rng,
    ).astype(np.int64)

print(f"Training pool: {len(train_idx_baseline)} / {n_train_full}")
print(f"Tumor/normal: {(y_train_full[train_idx_baseline] == 1).sum()} / "
      f"{(y_train_full[train_idx_baseline] == 0).sum()}")

class PCamTrainDataset(PCamImageDataset):
    """PCam training dataset with deterministic, label-preserving flips."""
    def __init__(self, h5_image_path, indices, labels, augment=False):
        super().__init__(h5_image_path, indices, labels)
        self.augment = augment

    def __getitem__(self, i):
        img_t, label = super().__getitem__(i)
        if self.augment:
            # Seed-controlled torch RNG makes the augmentation reproducible.
            if torch.rand(()) < 0.5:
                img_t = torch.flip(img_t, dims=[2])
            if torch.rand(()) < 0.5:
                img_t = torch.flip(img_t, dims=[1])
        return img_t, label

baseline_train_ds = PCamTrainDataset(
    pcam_files['train_x'], train_idx_baseline, y_train_full[train_idx_baseline], augment=True
)
baseline_train_loader = DataLoader(
    baseline_train_ds,
    batch_size=CONFIG['BASELINE_BATCH_SIZE'],
    shuffle=True,
    num_workers=CONFIG['BASELINE_NUM_WORKERS'],
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(CONFIG['BASELINE_NUM_WORKERS'] > 0),
)

print("✓ Fixed training pool and loaders prepared.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAIN + EVALUATE COMPACT ARCHITECTURE BASELINES ACROSS SEEDS
# ─────────────────────────────────────────────────────────────────────────────

BASELINE_DIR = os.path.join(OUT_DIR, 'cods_baselines')
os.makedirs(BASELINE_DIR, exist_ok=True)


def build_baseline(name, seed):
    """Create a task-specific 2-class model. ImageNet weights are optional."""
    set_seed(seed)
    try:
        # Do not pass img_size to generic CNNs such as ResNet18.
        # timm's ResNet constructor does not accept this argument; the
        # convolutional baselines support 96x96 inputs through their
        # adaptive/global pooling.
        model = timm.create_model(
            name,
            pretrained=CONFIG['BASELINE_PRETRAINED'],
            num_classes=2,
        )
    except Exception as e:
        print(f"  ⚠ pretrained={CONFIG['BASELINE_PRETRAINED']} unavailable for {name}: {e}")
        print("    Falling back to pretrained=False.")
        model = timm.create_model(
            name,
            pretrained=False,
            num_classes=2,
        )
    return model.to(DEVICE)


def evaluate_model_cpu_or_gpu(model, loader, desc):
    model.eval()
    probs_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc, leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            probs_all.append(probs)
            labels_all.append(labels.numpy())
    probs_all = np.concatenate(probs_all)
    labels_all = np.concatenate(labels_all)
    return {
        'auc': float(roc_auc_score(labels_all, probs_all)),
        'f1': float(f1_score(labels_all, probs_all >= 0.5, zero_division=0)),
        'acc': float(accuracy_score(labels_all, probs_all >= 0.5)),
        'probs': probs_all,
        'labels': labels_all,
    }


def train_one_baseline(model_name, seed):
    set_seed(seed)
    model = build_baseline(model_name, seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG['BASELINE_LR'],
        weight_decay=CONFIG['BASELINE_WEIGHT_DECAY'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG['BASELINE_EPOCHS']
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

    best_val_auc = -np.inf
    best_state = None
    history = []

    for epoch in range(1, CONFIG['BASELINE_EPOCHS'] + 1):
        model.train()
        running_loss = 0.0
        n_seen = 0
        for imgs, labels in tqdm(
            baseline_train_loader,
            desc=f"{model_name} seed={seed} epoch={epoch}/{CONFIG['BASELINE_EPOCHS']}",
            leave=False,
        ):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach()) * len(labels)
            n_seen += len(labels)

        scheduler.step()
        val_res = evaluate_model_cpu_or_gpu(
            model, val_loader, f"{model_name} seed={seed} val"
        )
        epoch_rec = {
            'epoch': epoch,
            'train_loss': running_loss / max(n_seen, 1),
            'val_auc': val_res['auc'],
            'val_f1': val_res['f1'],
        }
        history.append(epoch_rec)
        print(
            f"  epoch {epoch:02d}: loss={epoch_rec['train_loss']:.4f} "
            f"val_auc={epoch_rec['val_auc']:.4f}"
        )

        if val_res['auc'] > best_val_auc:
            best_val_auc = val_res['auc']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()

    # Evaluate test exactly once after validation-based checkpoint selection.
    test_res = evaluate_model_cpu_or_gpu(
        model, test_loader, f"{model_name} seed={seed} test"
    )

    # 95% bootstrap CI on test AUC -- sampling uncertainty on this fixed
    # test subset. Complements (does not replace) the cross-seed mean ± SD
    # computed later in cods_baseline_seed_summary.csv: that SD reflects
    # variance across different training runs; this CI reflects variance
    # from evaluating one checkpoint on one finite 5,000-image test sample.
    _, test_auc_ci_lo, test_auc_ci_hi = bootstrap_auc_ci(
        test_res['labels'], test_res['probs'], n_boot=1000, seed=seed
    )

    ckpt = os.path.join(BASELINE_DIR, f"{model_name}__seed{seed}.pt")
    torch.save(model.state_dict(), ckpt)
    with open(os.path.join(BASELINE_DIR, f"{model_name}__seed{seed}.json"), 'w') as f:
        json.dump({
            'model': model_name,
            'seed': int(seed),
            'best_val_auc': float(best_val_auc),
            'test_auc': float(test_res['auc']),
            'test_auc_ci_low': float(test_auc_ci_lo),
            'test_auc_ci_high': float(test_auc_ci_hi),
            'test_f1': float(test_res['f1']),
            'test_acc': float(test_res['acc']),
            'epochs': CONFIG['BASELINE_EPOCHS'],
            'train_subset': int(len(train_idx_baseline)),
            'history': history,
        }, f, indent=2)

    n_params = sum(p.numel() for p in model.parameters())
    size_mb = os.path.getsize(ckpt) / (1024 ** 2)
    return {
        'Model': model_name,
        'Seed': int(seed),
        'Best_Val_AUC': float(best_val_auc),
        'Test_AUC': float(test_res['auc']),
        'Test_AUC_CI_low': float(test_auc_ci_lo),
        'Test_AUC_CI_high': float(test_auc_ci_hi),
        'Test_F1': float(test_res['f1']),
        'Test_Acc': float(test_res['acc']),
        'Parameters': int(n_params),
        'Checkpoint_MB': float(size_mb),
        'Checkpoint': ckpt,
    }

baseline_records = []
for model_name in CONFIG['BASELINE_MODELS']:
    for seed in CONFIG['BASELINE_SEEDS']:
        print("\n" + "-" * 80)
        print(f"Training {model_name} | seed={seed}")
        baseline_records.append(train_one_baseline(model_name, seed))

baseline_seed_df = pd.DataFrame(baseline_records)
baseline_seed_df.to_csv(
    os.path.join(OUT_DIR, 'cods_baseline_per_seed.csv'), index=False
)
print("\n" + "=" * 80)
print("PER-SEED BASELINE RESULTS")
print("=" * 80)
print(baseline_seed_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SEED AGGREGATION + MOBILEVIT SEED ROBUSTNESS
# ─────────────────────────────────────────────────────────────────────────────

# Evaluate every available Stage-2 MobileViT checkpoint on the SAME frozen
# val/test subsets. Do not use test AUC to choose the golden checkpoint.
mobilevit_seed_records = []
for rec in seed_records:
    model = timm.create_model(CONFIG['BACKBONE'], pretrained=False, num_classes=2).to(DEVICE)
    state = torch.load(rec['deploy_pt'], map_location='cpu')
    model.load_state_dict(state, strict=True)
    model.eval()
    vr = evaluate_model_cpu_or_gpu(model, val_loader, f"MobileViT seed={rec['seed']} val")
    tr = evaluate_model_cpu_or_gpu(model, test_loader, f"MobileViT seed={rec['seed']} test")

    # 95% bootstrap CI on test AUC (sampling uncertainty on this fixed test
    # subset), same helper and rationale as the architecture-baseline cell.
    _, tr_auc_ci_lo, tr_auc_ci_hi = bootstrap_auc_ci(
        tr['labels'], tr['probs'], n_boot=1000, seed=int(rec['seed'])
    )

    mobilevit_seed_records.append({
        'Model': 'mobilevit_s',
        'Seed': int(rec['seed']),
        'Best_Val_AUC': float(vr['auc']),
        'Test_AUC': float(tr['auc']),
        'Test_AUC_CI_low': float(tr_auc_ci_lo),
        'Test_AUC_CI_high': float(tr_auc_ci_hi),
        'Test_F1': float(tr['f1']),
        'Test_Acc': float(tr['acc']),
        'Parameters': int(sum(p.numel() for p in model.parameters())),
        'Checkpoint_MB': float(os.path.getsize(rec['deploy_pt']) / (1024 ** 2)),
    })
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

mobilevit_seed_df = pd.DataFrame(mobilevit_seed_records)
mobilevit_seed_df.to_csv(os.path.join(OUT_DIR, 'mobilevit_seed_robustness.csv'), index=False)

all_seed_df = pd.concat([
    mobilevit_seed_df,
    baseline_seed_df[['Model','Seed','Best_Val_AUC','Test_AUC','Test_AUC_CI_low','Test_AUC_CI_high',
                       'Test_F1','Test_Acc','Parameters','Checkpoint_MB']]
], ignore_index=True)

summary_df = (
    all_seed_df.groupby('Model', as_index=False)
    .agg(
        Seeds=('Seed', 'count'),
        Val_AUC_mean=('Best_Val_AUC', 'mean'),
        Val_AUC_std=('Best_Val_AUC', 'std'),
        Test_AUC_mean=('Test_AUC', 'mean'),
        Test_AUC_std=('Test_AUC', 'std'),
        Test_F1_mean=('Test_F1', 'mean'),
        Test_F1_std=('Test_F1', 'std'),
        Parameters=('Parameters', 'first'),
        Checkpoint_MB=('Checkpoint_MB', 'first'),
    )
)

for c in ['Val_AUC_std','Test_AUC_std','Test_F1_std']:
    summary_df[c] = summary_df[c].fillna(0.0)

summary_df.to_csv(os.path.join(OUT_DIR, 'cods_baseline_seed_summary.csv'), index=False)

print("=" * 80)
print("CODS PAPER-FACING MEAN ± SD SUMMARY")
print("=" * 80)
for _, r in summary_df.iterrows():
    print(
        f"{r['Model']:<24} n={int(r['Seeds'])} | "
        f"Val AUC {r['Val_AUC_mean']:.4f}±{r['Val_AUC_std']:.4f} | "
        f"Test AUC {r['Test_AUC_mean']:.4f}±{r['Test_AUC_std']:.4f} | "
        f"F1 {r['Test_F1_mean']:.4f}±{r['Test_F1_std']:.4f} | "
        f"Params {int(r['Parameters']):,}"
    )

print("\nNOTE: Test_AUC_CI_low/Test_AUC_CI_high (in the per-seed CSVs, not this")
print("      mean±SD table) are 95% bootstrap CIs on a SINGLE checkpoint's test AUC --")
print("      sampling uncertainty from the fixed 5,000-image test subset. They are a")
print("      different quantity from the cross-seed Test_AUC_std shown above, which")
print("      reflects variance across independently trained models. Report both.")
print("\n✓ Saved per-seed and mean±SD tables.")
print("✓ Test set was not used for checkpoint selection; selection uses validation AUC.")

---
## NB4 — Precision Conversion & Accuracy Validation
Export the golden model to ONNX (FP32), then produce FP16 and INT8 PTQ variants.  
Validate AUC/F1 for all three before benchmarking.

In [ ]:
# ── Step A: Export FP32 ONNX ─────────────────────────────────────────────────
# NOTE: the legacy TorchScript-tracer exporter (dynamo=False) is prone to
# hanging on mobilevit_s: its fold/unfold patch<->token reshape logic combined
# with dynamic_axes on both batch dims can make symbolic shape tracing take
# an extremely long time. We try the newer torch.export-based exporter
# (dynamo=True) first — it traces via FX rather than the JIT tracer and
# handles this reshape-heavy control flow far more reliably — and only fall
# back to the legacy exporter if that path itself raises an error.
ONNX_FP32 = os.path.join(OUT_DIR, 'mobilevit_fp32.onnx')
dummy_input = torch.randn(1, 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']).to(DEVICE)

print('Exporting to ONNX (this can take a minute or two)...', flush=True)
t_export_start = time.time()

try:
    print('  Attempting dynamo-based exporter (torch.export)...', flush=True)
    # NOTE: don't force opset_version here. MobileViT's fold/unfold patch
    # logic includes a Resize op, and forcing a specific target opset makes
    # the dynamo exporter invoke ONNX's internal version converter, which
    # has no adapter to downgrade certain Resize configs to older opsets.
    # That conversion failure gets silently swallowed by the exporter and
    # leaves the graph in a state that later crashes ORT's symbolic shape
    # inference inside quant_pre_process. Passing opset_version=None lets
    # the exporter emit its own native (current, well-supported) opset
    # directly, skipping that conversion step entirely.
    torch.onnx.export(
        deploy_model,
        dummy_input,
        ONNX_FP32,
        export_params=True,
        opset_version=None,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        dynamo=True,
    )
    print(f'  ✓ dynamo export succeeded in {time.time() - t_export_start:.1f}s', flush=True)
except Exception as e:
    print(f'  ✗ dynamo export failed ({type(e).__name__}: {e}); falling back to legacy exporter...', flush=True)
    t_export_start = time.time()
    torch.onnx.export(
        deploy_model,
        dummy_input,
        ONNX_FP32,
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        dynamo=False,
    )
    print(f'  ✓ legacy export succeeded in {time.time() - t_export_start:.1f}s', flush=True)

# Validate ONNX graph
onnx_model = onnx.load(ONNX_FP32)
onnx.checker.check_model(onnx_model)
fp32_onnx_size_mb = os.path.getsize(ONNX_FP32) / (1024 ** 2)
print(f'✓ ONNX FP32 export complete: {ONNX_FP32}')
print(f'  Size: {fp32_onnx_size_mb:.2f} MB')

In [ ]:
# ── Step B: Convert to FP16 ONNX ─────────────────────────────────────────────
# onnxconverter_common.float16 converts all initializers and supported ops to fp16.
#
# CORRECTION (this cell was previously "fixed" incorrectly — see note below):
# an earlier pass added LayerNormalization/Softmax to op_block_list, on the
# theory that ORT lacks fast fp16 kernels for those ops and therefore inserts
# Cast(fp16<->fp32) around them. That reasoning was backwards. Blocking an op
# from fp16 conversion does NOT avoid Cast insertion — it forces it, because
# the op then sits in fp32 while everything around it is fp16, and ORT must
# bridge that boundary with explicit Cast nodes on every input and output.
# Measured directly on this model: blocking LayerNorm+Softmax produced 106
# Cast nodes; the plain default conversion (op_block_list=None) produces only
# 4 (just around the 2 Resize nodes, whose roi/scale inputs the ONNX spec
# requires to stay fp32 regardless of everything else). So the extra
# blocking was actively creating the Cast churn it was meant to prevent.
# The real cause of the originally-reported "FP16 is 4x slower" result was
# almost entirely the GPU benchmark's H2D/D2H transfer overhead (bug #2,
# fixed separately via IOBinding) — not fp16 op coverage. Reverting to the
# plain default conversion here.
ONNX_FP16 = os.path.join(OUT_DIR, 'mobilevit_fp16.onnx')
onnx_fp32 = onnx.load(ONNX_FP32)

onnx_fp16 = float16.convert_float_to_float16(
    onnx_fp32,
    keep_io_types=False,
)
onnx.save(onnx_fp16, ONNX_FP16)

# Sanity check: Cast nodes remaining in the fp16 graph should be small (just
# the ops on DEFAULT_OP_BLOCK_LIST that legitimately can't run in fp16, e.g.
# Resize's roi/scale inputs) — not the 100+ that blocking LayerNorm/Softmax
# produced.
n_cast = sum(1 for n in onnx_fp16.graph.node if n.op_type == 'Cast')
print(f'  Cast nodes in FP16 graph: {n_cast} (should be small — just ops on the default block list)')

fp16_size_mb = os.path.getsize(ONNX_FP16) / (1024 ** 2)
print(f'✓ ONNX FP16 export complete: {ONNX_FP16}')
print(f'  Size: {fp16_size_mb:.2f} MB  (compression: {fp32_size_mb / fp16_size_mb:.2f}x)')


In [ ]:
# ── Step B2: Shape-inference preprocessing for static quantization ───────────
# quantize_static expects a graph with shapes already inferred; skipping this
# entirely is what previously crashed ORT with a NameError further down.
#
# skip_symbolic_shape=True: ORT's SymbolicShapeInference raises an
# AssertionError inside _compute_conv_pool_shape on Conv nodes produced by
# the dynamo-based exporter (Step A) -- the symbolic solver doesn't
# recognize the dynamic-batch tensor rank this exporter emits. This model
# only has a dynamic batch dim (spatial size is fixed at CONFIG['IMG_SIZE']
# x CONFIG['IMG_SIZE']), so plain ONNX shape inference -- which still runs
# below regardless of this flag -- is sufficient for quantize_static; we
# don't need the symbolic solver's extra dynamic-dimension algebra here.
from onnxruntime.quantization.shape_inference import quant_pre_process

ONNX_FP32_PREP = os.path.join(OUT_DIR, 'mobilevit_fp32_prep.onnx')
quant_pre_process(
    input_model=ONNX_FP32,
    output_model_path=ONNX_FP32_PREP,
    skip_symbolic_shape=True,
)
print(f'✓ Preprocessed FP32 model for quantization: {ONNX_FP32_PREP}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step C2: SECOND-GENERATION MIXED-PRECISION INT8 PTQ
#
# Goal:
#   Recover accuracy from the first Conv-only INT8 attempt while still getting
#   useful CPU-side INT8 acceleration.
#
# Strategy:
#   • Keep transformer/attention-sensitive computation FP32.
#   • Quantize only a selected subset of Conv nodes.
#   • Compare TWO candidates on the validation set later:
#       1) SAFE_CONV  = Conv nodes that do not directly feed sensitive ops
#       2) STEM_CONV  = early/front Conv nodes only
#   • The best candidate is selected using VALIDATION AUC only.
#   • The selected node set is then re-quantized using the full calibration set.
#
# This avoids choosing a quantization strategy using the test set.
# ─────────────────────────────────────────────────────────────────────────────

from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    QuantFormat,
    CalibrationMethod,
    CalibrationDataReader,
)
import onnx
from collections import defaultdict

ONNX_INT8 = os.path.join(OUT_DIR, "mobilevit_int8_mixed_precision.onnx")
CANDIDATE_DIR = os.path.join(OUT_DIR, "mixed_precision_candidates")
os.makedirs(CANDIDATE_DIR, exist_ok=True)

print("=" * 80)
print("SECOND-GENERATION MIXED-PRECISION INT8 PTQ")
print("=" * 80)
print("FP32 model :", ONNX_FP32_PREP)
print("Final INT8 :", ONNX_INT8)
print("Format     : QDQ")
print("Per-channel: False")
print("Sensitive ops kept FP32: MatMul, Gemm, LayerNorm, Softmax, Sigmoid, etc.")
print()

if "calib_idx" not in globals():
    raise RuntimeError(
        "calib_idx is not defined. Run the dataset/calibration selection cell first."
    )

calib_indices = np.asarray(
    calib_idx[:CONFIG["CALIB_IMAGES"]], dtype=np.int64
)
if len(calib_indices) == 0:
    raise RuntimeError("No calibration indices were selected.")


class PCamCalibrationReader(CalibrationDataReader):
    """One-sample-at-a-time reader to avoid ORT Entropy collector shape errors."""
    def __init__(self, image_path, indices, input_name="input"):
        self.image_path = image_path
        self.indices = np.asarray(indices, dtype=np.int64)
        self.input_name = input_name
        self._pos = 0
        self._h5 = None
        self._images = None

    def _open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.image_path, "r")
            self._images = self._h5[list(self._h5.keys())[0]]

    def get_next(self):
        if self._pos >= len(self.indices):
            return None
        self._open()
        idx = int(self.indices[self._pos])
        img = np.asarray(self._images[idx], dtype=np.float32) / 255.0
        if img.ndim != 3:
            raise ValueError(f"Unexpected calibration image shape: {img.shape}")
        img = np.transpose(img, (2, 0, 1))[None, ...]
        img = np.ascontiguousarray(img, dtype=np.float32)
        expected = (1, 3, CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"])
        if img.shape != expected:
            raise ValueError(f"Calibration tensor {img.shape}; expected {expected}")
        self._pos += 1
        return {self.input_name: img}

    def rewind(self):
        self._pos = 0

    def __del__(self):
        try:
            if self._h5 is not None:
                self._h5.close()
        except Exception:
            pass


def make_calib_reader(indices):
    return PCamCalibrationReader(
        pcam_files["valid_x"], indices, input_name="input"
    )


# ─────────────────────────────────────────────────────────────────────────────
# Discover Conv nodes and identify Conv nodes immediately adjacent to
# transformer/attention-sensitive operations.
# ─────────────────────────────────────────────────────────────────────────────

graph_model = onnx.load(ONNX_FP32_PREP)
nodes = list(graph_model.graph.node)
conv_nodes = [n for n in nodes if n.op_type == "Conv"]

consumer_ops = defaultdict(list)
for n in nodes:
    for inp in n.input:
        consumer_ops[inp].append(n.op_type)

SENSITIVE_OPS = {
    "MatMul", "Gemm", "LayerNormalization", "Softmax",
    "Sigmoid", "Tanh", "Exp", "Log", "Pow", "Sqrt",
    "ReduceMean", "ReduceSum", "Div"
}

safe_conv_nodes = []
for n in conv_nodes:
    direct_consumers = []
    for out in n.output:
        direct_consumers.extend(consumer_ops.get(out, []))
    if not any(op in SENSITIVE_OPS for op in direct_consumers):
        safe_conv_nodes.append(n.name)

# Front/stem candidate: quantize only the earliest ~35% of Conv nodes.
# This deliberately keeps most MobileViT transformer-adjacent computation FP32.
front_n = max(1, int(np.ceil(len(conv_nodes) * 0.35)))
stem_conv_nodes = [n.name for n in conv_nodes[:front_n]]

# If names are empty (legal in some ONNX graphs), use unique node indices as
# a fallback by selecting the actual node objects through their generated IDs.
def ensure_names(selected_names, label):
    if selected_names and all(selected_names):
        return selected_names
    raise RuntimeError(f"Could not obtain stable ONNX node names for {label} candidate.")

safe_conv_nodes = ensure_names(safe_conv_nodes, "SAFE_CONV") if safe_conv_nodes else []
stem_conv_nodes = ensure_names(stem_conv_nodes, "STEM_CONV")

candidate_specs = {
    "SAFE_CONV": safe_conv_nodes,
    "STEM_CONV": stem_conv_nodes,
}

# If the graph has no directly-safe Conv nodes, use the front candidate only.
if not safe_conv_nodes:
    candidate_specs = {"STEM_CONV": stem_conv_nodes}

print(f"Total Conv nodes : {len(conv_nodes)}")
print(f"Safe Conv nodes  : {len(safe_conv_nodes)}")
print(f"Stem Conv nodes  : {len(stem_conv_nodes)} ({front_n}/{len(conv_nodes)})")
print("Candidates       :", {k: len(v) for k, v in candidate_specs.items()})


# ─────────────────────────────────────────────────────────────────────────────
# Quantization helper.
# Screening uses 300 calibration images to keep the experiment practical.
# The winner is re-quantized later with the complete configured calibration set.
# ─────────────────────────────────────────────────────────────────────────────

SCREEN_CALIB_N = min(300, len(calib_indices))
screen_indices = calib_indices[:SCREEN_CALIB_N]


def quantize_nodes(node_names, output_path, calibration_indices, method=CalibrationMethod.MinMax):
    if os.path.exists(output_path):
        os.remove(output_path)

    reader = make_calib_reader(calibration_indices)

    quantize_static(
        model_input=ONNX_FP32_PREP,
        model_output=output_path,
        calibration_data_reader=reader,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
        quant_format=QuantFormat.QDQ,
        per_channel=False,
        calibrate_method=method,
        nodes_to_quantize=list(node_names),
        op_types_to_quantize=["Conv"],
    )

    if not os.path.exists(output_path):
        raise RuntimeError(f"Quantization failed to create {output_path}")


candidate_paths = {}
for candidate_name, node_names in candidate_specs.items():
    path = os.path.join(
        CANDIDATE_DIR,
        f"mobilevit_{candidate_name.lower()}_int8_screen.onnx"
    )
    print(f"\nQuantizing {candidate_name}: {len(node_names)} Conv nodes")
    t0 = time.time()
    quantize_nodes(node_names, path, screen_indices, CalibrationMethod.MinMax)
    candidate_paths[candidate_name] = path
    print(f"✓ {candidate_name}: {time.time() - t0:.1f}s | {os.path.getsize(path)/(1024**2):.2f} MB")

print("\n✓ Candidate models created. Validation-only selection happens in the next cell.")


In [ ]:
# ── Evaluate all three ONNX variants via ONNX Runtime ─────────────────────────
# FP32 and FP16 run on CUDAExecutionProvider (T4 GPU).
# INT8 runs on CPUExecutionProvider (simulates resource-constrained CPU inference).

def get_ort_session(onnx_path, use_gpu=True):
    providers = (
        ['CUDAExecutionProvider', 'CPUExecutionProvider'] if use_gpu
        else ['CPUExecutionProvider']
    )
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.log_severity_level = 3  # suppress warnings
    return ort.InferenceSession(onnx_path, sess_options=sess_opts, providers=providers)


def evaluate_onnx(onnx_path, loader, use_gpu=True, desc='ONNX Eval'):
    sess       = get_ort_session(onnx_path, use_gpu)
    input_name = sess.get_inputs()[0].name
    all_probs, all_labels = [], []

    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs_np = imgs.numpy()  # (B, 3, H, W) float32

        # FP16 models expect float16 inputs
        if 'fp16' in os.path.basename(onnx_path):
            imgs_np = imgs_np.astype(np.float16)

        logits = sess.run(None, {input_name: imgs_np})[0]  # (B, 2)

        # Softmax in numpy (works for both float32 and float16 outputs)
        logits = logits.astype(np.float32)
        exp_l  = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs  = (exp_l / exp_l.sum(axis=1, keepdims=True))[:, 1]

        all_probs.append(probs)
        all_labels.append(labels.numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs)
    f1  = f1_score(all_labels, (all_probs > 0.5).astype(int))
    acc = accuracy_score(all_labels, (all_probs > 0.5).astype(int))
    return {'auc': auc, 'f1': f1, 'acc': acc, 'probs': all_probs, 'labels': all_labels}



# ─────────────────────────────────────────────────────────────────────────────
# SECOND-GEN CANDIDATE SELECTION — VALIDATION ONLY
# ─────────────────────────────────────────────────────────────────────────────
# Never use the test set to select a quantization strategy.
# The candidate with the highest validation AUC wins. The winning node set is
# then re-quantized below with the FULL calibration set before test evaluation.

print("\n" + "=" * 80)
print("MIXED-PRECISION CANDIDATE SELECTION")
print("=" * 80)

candidate_val_scores = {}
for candidate_name, candidate_path in candidate_paths.items():
    try:
        r = evaluate_onnx(
            candidate_path,
            val_loader,
            use_gpu=False,
            desc=f"Val {candidate_name}",
        )
        candidate_val_scores[candidate_name] = r["auc"]
        print(f"{candidate_name:<12} validation AUC: {r['auc']:.4f}")
    except Exception as e:
        candidate_val_scores[candidate_name] = -np.inf
        print(f"{candidate_name:<12} FAILED: {type(e).__name__}: {e}")

if not any(np.isfinite(v) for v in candidate_val_scores.values()):
    raise RuntimeError("All mixed-precision INT8 candidates failed validation.")

selected_candidate = max(candidate_val_scores, key=candidate_val_scores.get)
selected_nodes = candidate_specs[selected_candidate]
selected_screen_auc = candidate_val_scores[selected_candidate]

print(f"\n✓ Selected candidate: {selected_candidate}")
print(f"  Screening validation AUC: {selected_screen_auc:.4f}")
print(f"  Conv nodes quantized: {len(selected_nodes)}")

# Re-quantize the selected node set with the COMPLETE calibration set.
print(f"\nRe-quantizing {selected_candidate} with {len(calib_indices)} calibration images...")
t0 = time.time()
quantize_nodes(
    selected_nodes,
    ONNX_INT8,
    calib_indices,
    CalibrationMethod.MinMax,
)
print(f"✓ Final mixed-precision INT8 created in {time.time() - t0:.1f}s")

# BUGFIX: int8_size_mb was never assigned anywhere in the notebook (only
# printed inline below), but the results-table cell's add_row('MobileViT
# INT8', ...) call reads it as a module-level variable -> NameError at
# execution. fp32_size_mb and fp16_size_mb are both set immediately after
# their files are written (see the FP32 export and FP16 conversion cells);
# do the same here for INT8 so the variable exists downstream.
int8_size_mb = os.path.getsize(ONNX_INT8) / (1024 ** 2)

print(f"  Path: {ONNX_INT8}")
print(f"  Size: {int8_size_mb:.2f} MB")
print("  Calibration: MinMax")
print("  Calibration images:", len(calib_indices))


# Detect available execution providers
#
# BUGFIX: `ort.get_available_providers()` only reports which EPs the installed
# wheel was *compiled with* — it does not verify the EP's native libraries
# (CUDA/cuDNN/cuBLAS .so files) actually load at runtime. On a CUDA-version
# mismatch (e.g. an onnxruntime-gpu wheel built for CUDA 13 running on a
# CUDA 12.x host), 'CUDAExecutionProvider' still shows up in this list, but
# `InferenceSession` silently falls back to CPUExecutionProvider the moment
# it actually tries to load it — so GPU_AVAIL was True and eval numbers
# above were quietly computed on CPU with no error at all. The fix is to
# probe it directly: build a session requesting CUDA only and check what
# provider it actually initialized with.
avail_eps = ort.get_available_providers()
GPU_AVAIL = False
if 'CUDAExecutionProvider' in avail_eps:
    try:
        _probe_opts = ort.SessionOptions()
        _probe_opts.log_severity_level = 3
        _probe_sess = ort.InferenceSession(
            ONNX_FP32, sess_options=_probe_opts, providers=['CUDAExecutionProvider']
        )
        GPU_AVAIL = _probe_sess.get_providers()[0] == 'CUDAExecutionProvider'
        del _probe_sess
    except Exception as e:
        print(f'  CUDA EP probe failed: {type(e).__name__}: {str(e)[:200]}')
        GPU_AVAIL = False

print(f'Available ORT providers (compiled-in): {avail_eps}')
print(f'CUDA EP actually usable (runtime-probed): {GPU_AVAIL}')
print(f'GPU inference: {"enabled" if GPU_AVAIL else "not available, falling back to CPU"}')

if not GPU_AVAIL and torch.cuda.is_available():
    print('\n⚠ WARNING: torch.cuda.is_available() is True (a GPU IS attached) but '
          'the CUDA execution provider failed to initialize in onnxruntime.')
    print('  Two known causes, in order of likelihood:')
    print('  1. onnxruntime-gpu/CUDA version mismatch (e.g. a CUDA-13 wheel on a')
    print('     CUDA-12.x host) — missing libcublasLt.so.13 / cuDNN 9.x for CUDA 13.')
    print('     Fix: restart kernel, ensure §0 installed `onnxruntime-gpu<1.27`,')
    print('     then re-run from the top.')
    print('  2. Both `onnxruntime` and `onnxruntime-gpu` got installed and one')
    print('     shadowed the other (§0 uninstalls both before reinstalling).')
    print('  Continuing on CPU for now — GPU rows in the final results table will be blank.')

# Evaluate
print('\n--- Evaluating ONNX FP32 (GPU) ---')
onnx_fp32_val  = evaluate_onnx(ONNX_FP32, val_loader,  use_gpu=GPU_AVAIL, desc='FP32 Val')
onnx_fp32_test = evaluate_onnx(ONNX_FP32, test_loader, use_gpu=GPU_AVAIL, desc='FP32 Test')

print('\n--- Evaluating ONNX FP16 (GPU) ---')
onnx_fp16_val  = evaluate_onnx(ONNX_FP16, val_loader,  use_gpu=GPU_AVAIL, desc='FP16 Val')
onnx_fp16_test = evaluate_onnx(ONNX_FP16, test_loader, use_gpu=GPU_AVAIL, desc='FP16 Test')

print('\n--- Evaluating ONNX INT8 (CPU) ---')
onnx_int8_val  = evaluate_onnx(ONNX_INT8, val_loader,  use_gpu=False, desc='INT8 Val')
onnx_int8_test = evaluate_onnx(ONNX_INT8, test_loader, use_gpu=False, desc='INT8 Test')

# Summary
print('\n=== Accuracy Validation Summary ===')
print(f'{"Variant":<12} {"Val AUC":>8} {"Val F1":>8} {"Test AUC":>10} {"Test F1":>8}')
print('-' * 52)
for name, v, t in [
    ('FP32',  onnx_fp32_val,  onnx_fp32_test),
    ('FP16',  onnx_fp16_val,  onnx_fp16_test),
    ('INT8',  onnx_int8_val,  onnx_int8_test),
]:
    print(f'{name:<12} {v["auc"]:>8.4f} {v["f1"]:>8.4f} {t["auc"]:>10.4f} {t["f1"]:>8.4f}')

# Check for significant INT8 degradation (warn if ΔTest AUC > 0.01)
delta_int8_test = onnx_fp32_test['auc'] - onnx_int8_test['auc']
if delta_int8_test > 0.01:
    print(f'\n⚠ INT8 degradation on test: ΔAUC = {delta_int8_test:.4f} > 0.01.')
    print('  Consider increasing CALIB_IMAGES to 500 before accepting this result.')
else:
    print(f'\n✓ INT8 degradation acceptable: ΔAUC = {delta_int8_test:.4f} (threshold 0.01)')

---
### Threshold calibration (bugfix)
`F1` was previously computed with a fixed `prob > 0.5` threshold for every precision variant. Post-training quantization (and, to a lesser extent, FP16 rounding) shifts the *shape* of the output probability distribution even when ranking (AUC) barely moves — so a threshold tuned for FP32's distribution is not necessarily still near-optimal for INT8's. Comparing F1 at a fixed 0.5 threshold conflates threshold miscalibration with real capability loss. Here we pick each variant's threshold on the **validation** set (maximizing F1) and apply that fixed threshold to the **test** set, so the calibrated F1 isolates how much of the drop is recoverable by recalibrating vs. how much is real.

In [ ]:
# ── Per-variant threshold calibration ─────────────────────────────────────
# For each precision variant: sweep thresholds on VAL probs to find the one
# that maximizes F1, then apply that fixed threshold to TEST probs. This
# gives a fair 'best F1 this variant can reach' number, separate from the
# fixed-0.5 number (which the notebook still reports for comparison).
#
# Also reports sensitivity (recall on the positive/tumor class, i.e. TPR --
# the fraction of actual positives correctly caught) and specificity (TNR --
# the fraction of actual negatives correctly cleared), at both the fixed 0.5
# threshold and the calibrated threshold, since F1/AUC alone can hide which
# error type (missed tumors vs. false alarms) a variant is actually trading
# off. PCam labels: 1 = tumor patch, 0 = normal patch.

def best_f1_threshold(val_probs, val_labels, n_steps=200):
    """Return the threshold in (0, 1) that maximizes F1 on the given val set."""
    thresholds = np.linspace(0.01, 0.99, n_steps)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        preds = (val_probs > t).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)


def sens_spec(labels, probs, threshold):
    """Sensitivity (TPR, recall on class 1) and specificity (TNR, recall on
    class 0) at a given threshold. confusion_matrix(labels=[0, 1]) fixes
    row/column order so this doesn't silently break if a batch has only
    one class present."""
    preds = (probs > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    return float(sensitivity), float(specificity)


calibrated_f1 = {}

for name, val_res, test_res in [
    ('FP32', onnx_fp32_val, onnx_fp32_test),
    ('FP16', onnx_fp16_val, onnx_fp16_test),
    ('INT8', onnx_int8_val, onnx_int8_test),
]:
    thresh, val_f1_at_thresh = best_f1_threshold(val_res['probs'], val_res['labels'])
    test_preds_calibrated = (test_res['probs'] > thresh).astype(int)
    test_f1_calibrated = f1_score(test_res['labels'], test_preds_calibrated)
    test_f1_fixed = f1_score(test_res['labels'], (test_res['probs'] > 0.5).astype(int))

    sens_fixed, spec_fixed = sens_spec(test_res['labels'], test_res['probs'], 0.5)
    sens_calib, spec_calib = sens_spec(test_res['labels'], test_res['probs'], thresh)

    calibrated_f1[name] = {
        'threshold':      thresh,
        'val_f1':         val_f1_at_thresh,
        'test_f1_fixed':  test_f1_fixed,
        'test_f1':        test_f1_calibrated,
        'sens_fixed':     sens_fixed,
        'spec_fixed':     spec_fixed,
        'sens_calib':     sens_calib,
        'spec_calib':     spec_calib,
    }

print(f'{"Variant":<8} {"Threshold":>10} {"F1 @ 0.5":>10} {"F1 @ calib":>11} {"Recovered":>10}')
print('-' * 54)
for name, d in calibrated_f1.items():
    recovered = d['test_f1'] - d['test_f1_fixed']
    print(f"{name:<8} {d['threshold']:>10.3f} {d['test_f1_fixed']:>10.4f} "
          f"{d['test_f1']:>11.4f} {recovered:>+10.4f}")

print('\nNOTE: threshold chosen on VAL, applied to TEST (no test-set leakage).')
print('      Any gap still remaining after recalibration is real degradation,')
print('      not a threshold artifact.')

print(f'\n{"Variant":<8} {"Sens @ 0.5":>11} {"Spec @ 0.5":>11} {"Sens @ calib":>13} {"Spec @ calib":>13}')
print('-' * 60)
for name, d in calibrated_f1.items():
    print(f"{name:<8} {d['sens_fixed']:>11.4f} {d['spec_fixed']:>11.4f} "
          f"{d['sens_calib']:>13.4f} {d['spec_calib']:>13.4f}")

print('\nNOTE: sensitivity = recall on tumor-positive class (missed-tumor risk if low);')
print('      specificity = recall on normal-negative class (false-alarm risk if low).')
print('      Calibrating the threshold for F1 shifts the sensitivity/specificity')
print('      trade-off too -- check both, not just F1, before picking a variant for')
print('      a use case where missed tumors and false alarms are not equally costly.')


---
## NB5 — Controlled T4 Efficiency Benchmark
Rigorous `batch=1` latency using CUDA Events (GPU) and `time.perf_counter` (CPU).  
**Protocol:** 50 warm-up iterations → 1000 timed iterations → median + p95.

In [ ]:
def benchmark_onnx_gpu(onnx_path, input_np_fp32, warmup=50, iters=1000):
    """GPU latency via CUDA Events + IOBinding. Returns dict of timing stats in ms.

    BUGFIX: the original version called `sess.run(None, {input_name: numpy_array})`
    inside the timed loop. Passing a numpy (host) array to `sess.run` forces ORT to
    copy it host->device on every single call, and copy the output device->host
    before returning — both inside the CUDA-Event-timed window. For a 5M-param
    model at batch=1, that H2D/D2H copy overhead is comparable to (or larger than)
    the actual compute, which is exactly why FP32 GPU (7.02ms) and FP32 CPU
    (6.97ms) came out nearly identical: the "GPU" number was mostly transfer
    overhead, not GPU compute.

    Fix: use ORT's IOBinding to pre-allocate the input as a CUDA OrtValue once
    (outside the loop) and bind the output to GPU memory too. `sess.run_with_iobinding`
    then does not touch host memory at all during the timed iterations — it's a
    compute-only measurement, which is what "GPU latency" is supposed to mean.
    """
    assert GPU_AVAIL, 'GPU not available — use benchmark_onnx_cpu instead.'
    sess        = get_ort_session(onnx_path, use_gpu=True)
    input_name  = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name

    np_dtype = np.float16 if 'fp16' in os.path.basename(onnx_path) else np.float32
    inp = input_np_fp32.astype(np_dtype)

    # Pre-allocate input directly on the GPU (device_id=0 matches Kaggle's single-T4).
    io_binding = sess.io_binding()
    input_ortvalue = ort.OrtValue.ortvalue_from_numpy(inp, 'cuda', 0)
    io_binding.bind_ortvalue_input(input_name, input_ortvalue)
    # bind_output with a device (no numpy buffer) keeps the result on GPU too,
    # so the timed region never triggers a device->host copy either.
    io_binding.bind_output(output_name, 'cuda', 0)

    # Warm-up (also lets CUDA graphs / cuDNN autotuning settle before timing).
    for _ in range(warmup):
        sess.run_with_iobinding(io_binding)

    # CUDA-event timed loop — compute only, no host<->device transfer inside it.
    timings = []
    for _ in range(iters):
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        torch.cuda.synchronize()
        start_evt.record()
        sess.run_with_iobinding(io_binding)
        end_evt.record()
        torch.cuda.synchronize()
        timings.append(start_evt.elapsed_time(end_evt))

    timings = np.array(timings)
    return {
        'median_ms':     float(np.median(timings)),
        'p95_ms':        float(np.percentile(timings, 95)),
        'mean_ms':       float(np.mean(timings)),
        'throughput_fps': float(1000.0 / np.median(timings)),
    }


def benchmark_onnx_cpu(onnx_path, input_np_fp32, warmup=50, iters=1000):
    """CPU latency via perf_counter. Returns dict of timing stats in ms.

    No IOBinding fix needed here: CPU inference already operates on host
    memory, so there's no H2D/D2H transfer to hide — sess.run() with a numpy
    array is a fair CPU measurement as-is.
    """
    sess       = get_ort_session(onnx_path, use_gpu=False)
    input_name = sess.get_inputs()[0].name
    inp        = input_np_fp32.astype(np.float32)

    for _ in range(warmup):
        sess.run(None, {input_name: inp})

    timings = []
    for _ in range(iters):
        t0 = time.perf_counter()
        sess.run(None, {input_name: inp})
        timings.append((time.perf_counter() - t0) * 1000.0)

    timings = np.array(timings)
    return {
        'median_ms':      float(np.median(timings)),
        'p95_ms':         float(np.percentile(timings, 95)),
        'mean_ms':        float(np.mean(timings)),
        'throughput_fps': float(1000.0 / np.median(timings)),
    }


# Single real image for benchmarking (batch=1)
bench_img_np = sample_imgs[0:1].cpu().numpy()  # (1, 3, 96, 96) float32

print(f'Benchmark protocol: {CONFIG["WARMUP_ITERS"]} warm-up / {CONFIG["BENCH_ITERS"]} measurement @ batch=1')
print(f'Image shape: {bench_img_np.shape}  dtype: {bench_img_np.dtype}')
print('GPU timing uses IOBinding (compute-only, no H2D/D2H in the timed loop).')


In [ ]:
# ── Run all benchmarks ─────────────────────────────────────────────────────────
# BUGFIX: GPU_AVAIL is now runtime-probed (see previous cell), so this branch
# should only be entered when CUDA actually works. As defense in depth, GPU
# calls are still wrapped in try/except: IOBinding's ortvalue_from_numpy has
# no silent-fallback behavior like InferenceSession does, so any residual
# CUDA/driver hiccup here raises instead of quietly degrading — without the
# guard that kills the whole papermill run and loses the CPU/INT8 results
# that come after it. On failure we log it, leave the GPU rows out of
# bench_results, and keep going.
W = CONFIG['WARMUP_ITERS']
N = CONFIG['BENCH_ITERS']

bench_results = {}

if GPU_AVAIL:
    try:
        print('Benchmarking FP32 on GPU...')
        bench_results['FP32_GPU'] = benchmark_onnx_gpu(ONNX_FP32, bench_img_np, W, N)
        print(f'  FP32 GPU: {bench_results["FP32_GPU"]["median_ms"]:.3f} ms median')

        print('Benchmarking FP16 on GPU...')
        bench_results['FP16_GPU'] = benchmark_onnx_gpu(ONNX_FP16, bench_img_np, W, N)
        print(f'  FP16 GPU: {bench_results["FP16_GPU"]["median_ms"]:.3f} ms median')
    except Exception as e:
        print(f'\n⚠ GPU benchmarking failed despite the CUDA EP probe passing: '
              f'{type(e).__name__}: {str(e)[:200]}')
        print('  Continuing without GPU benchmark rows — see results table for what')
        print('  was measured. This usually means a CUDA/cuDNN library mismatch;')
        print('  check that `onnxruntime-gpu<1.27` was installed in §0.')

print('Benchmarking INT8 on CPU...')
bench_results['INT8_CPU'] = benchmark_onnx_cpu(ONNX_INT8, bench_img_np, W, N)
print(f'  INT8 CPU: {bench_results["INT8_CPU"]["median_ms"]:.3f} ms median')

print('Benchmarking FP32 on CPU (reference)...')
bench_results['FP32_CPU'] = benchmark_onnx_cpu(ONNX_FP32, bench_img_np, W, N)
print(f'  FP32 CPU: {bench_results["FP32_CPU"]["median_ms"]:.3f} ms median')

print('\n=== Benchmark Summary ===')
print(f'{"Variant":<12} {"Median ms":>10} {"p95 ms":>8} {"FPS":>10}')
print('-' * 44)
for k, v in bench_results.items():
    print(f'{k:<12} {v["median_ms"]:>10.3f} {v["p95_ms"]:>8.3f} {v["throughput_fps"]:>10.1f}')

if 'FP32_GPU' not in bench_results:
    print('\nNOTE: no GPU rows in bench_results — Speedup_x_same_device for FP16 and')
    print('      Speedup_x_vs_GPU_FP32 for all variants will be None downstream.')


In [ ]:
# ── Compile and save full results table ──────────────────────────────────────
FP32_REF_AUC     = onnx_fp32_test['auc']
FP32_GPU_REF_LAT = bench_results.get('FP32_GPU', {}).get('median_ms', None)
FP32_CPU_REF_LAT = bench_results.get('FP32_CPU', {}).get('median_ms', None)

# BUGFIX: the original code compared every variant's Speedup_x against a single
# GPU FP32 baseline (FP32_REF_LAT), even for INT8, which only ever runs on CPU.
# That's a device confound: INT8's 0.37x "speedup" was really just measuring
# "CPU is slower than GPU," not "INT8 quantization is slow." Each variant is
# now compared against the FP32 baseline on *its own* device, so Speedup_x
# isolates the precision effect. A separate cross-device column is kept too,
# clearly labeled, for readers who want the GPU-vs-CPU deployment picture.
results_table = []

def add_row(name, precision, device, size_mb, bench_key, auc_test, f1_test, ref_auc, calib,
            test_probs=None, test_labels=None):
    b = bench_results.get(bench_key, {})
    lat = b.get('median_ms', None)
    fps = b.get('throughput_fps', None)
    p95 = b.get('p95_ms', None)

    same_device_ref_lat = FP32_GPU_REF_LAT if device == 'T4 GPU' else FP32_CPU_REF_LAT

    # 95% bootstrap CI on test AUC (sampling uncertainty on the fixed 5,000-
    # image test subset), using the same helper defined earlier in the
    # notebook. Optional because add_row's ref_auc-only callers (none
    # currently) shouldn't be forced to supply probs/labels.
    if test_probs is not None and test_labels is not None:
        _, auc_ci_lo, auc_ci_hi = bootstrap_auc_ci(test_labels, test_probs, n_boot=1000, seed=0)
    else:
        auc_ci_lo, auc_ci_hi = None, None

    results_table.append({
        'Variant':                name,
        'Precision':              precision,
        'Device':                 device,
        'Size_MB':                round(size_mb, 2),
        'Compression_x':          round(fp32_size_mb / size_mb, 2),
        'Test_AUC':               round(auc_test, 4),
        'AUC_CI_low':             round(auc_ci_lo, 4) if auc_ci_lo is not None and not np.isnan(auc_ci_lo) else None,
        'AUC_CI_high':            round(auc_ci_hi, 4) if auc_ci_hi is not None and not np.isnan(auc_ci_hi) else None,
        'AUC_Retention_pct':      round(auc_test / ref_auc * 100, 2),
        'Delta_AUC':              round(auc_test - ref_auc, 4),
        'F1_fixed_thresh_0.5':    round(f1_test, 4),
        'F1_calibrated':          round(calib['test_f1'], 4),
        'Calib_Threshold':        round(calib['threshold'], 4),
        # Sensitivity = recall on tumor-positive class (1); Specificity = recall
        # on normal-negative class (0). Reported at both the fixed 0.5 threshold
        # (matches F1_fixed_thresh_0.5) and the calibrated threshold (matches
        # F1_calibrated), since a variant can trade sensitivity for specificity
        # (or vice versa) even when F1/AUC alone look similar.
        'Sensitivity_fixed_0.5':  round(calib['sens_fixed'], 4),
        'Specificity_fixed_0.5':  round(calib['spec_fixed'], 4),
        'Sensitivity_calibrated': round(calib['sens_calib'], 4),
        'Specificity_calibrated': round(calib['spec_calib'], 4),
        'Median_ms':              round(lat, 3) if lat else None,
        'p95_ms':                 round(p95, 3) if p95 else None,
        'FPS':                    round(fps, 1) if fps else None,
        'Speedup_x_same_device':  round(same_device_ref_lat / lat, 2) if (lat and same_device_ref_lat) else None,
        'Speedup_x_vs_GPU_FP32':  round(FP32_GPU_REF_LAT / lat, 2) if (lat and FP32_GPU_REF_LAT) else None,
    })

add_row('MobileViT FP32',  'FP32', 'T4 GPU', fp32_size_mb, 'FP32_GPU',
        onnx_fp32_test['auc'], onnx_fp32_test['f1'], FP32_REF_AUC, calibrated_f1['FP32'],
        test_probs=onnx_fp32_test['probs'], test_labels=onnx_fp32_test['labels'])

add_row('MobileViT FP16',  'FP16', 'T4 GPU', fp16_size_mb, 'FP16_GPU',
        onnx_fp16_test['auc'], onnx_fp16_test['f1'], FP32_REF_AUC, calibrated_f1['FP16'],
        test_probs=onnx_fp16_test['probs'], test_labels=onnx_fp16_test['labels'])

add_row('MobileViT INT8',  'INT8', 'CPU',    int8_size_mb, 'INT8_CPU',
        onnx_int8_test['auc'], onnx_int8_test['f1'], FP32_REF_AUC, calibrated_f1['INT8'],
        test_probs=onnx_int8_test['probs'], test_labels=onnx_int8_test['labels'])

# Also add CPU FP32 as a reference row if available
if 'FP32_CPU' in bench_results:
    add_row('MobileViT FP32 (CPU)', 'FP32', 'CPU', fp32_size_mb, 'FP32_CPU',
            onnx_fp32_test['auc'], onnx_fp32_test['f1'], FP32_REF_AUC, calibrated_f1['FP32'],
            test_probs=onnx_fp32_test['probs'], test_labels=onnx_fp32_test['labels'])

df_results = pd.DataFrame(results_table)
df_results.to_csv(os.path.join(OUT_DIR, 'results_table.csv'), index=False)
print(df_results.to_string(index=False))
print(f'\n✓ Results table saved to {OUT_DIR}/results_table.csv')
print('\nNOTE: Speedup_x_same_device compares each variant to the FP32 baseline')
print('      on the SAME device (GPU-vs-GPU for FP16, CPU-vs-CPU for INT8) —')
print('      use this for judging precision effects. Speedup_x_vs_GPU_FP32 is')
print('      kept only for the cross-device deployment picture and will always')
print('      make CPU rows look artificially slow.')
print('\nNOTE: Sensitivity/Specificity are reported at both the fixed 0.5 threshold')
print('      and the calibrated threshold — a variant can shift the balance between')
print('      missed-tumor risk (low sensitivity) and false-alarm risk (low')
print('      specificity) even when F1/AUC alone look similar.')
print('\nNOTE: AUC_CI_low/AUC_CI_high are 95% percentile bootstrap CIs on Test_AUC')
print('      (1000 resamples), capturing sampling uncertainty from the fixed')
print('      5,000-image test subset -- report alongside the point estimate.')


## CODS 2026 — Paper-Facing Comparison

Use the following tables to report **mean ± SD across seeds**, not the single best seed.
The precision/latency table below remains the deployment study; the architecture table is
the task-performance baseline study.

In [ ]:
print("=" * 100)
print("TABLE — ARCHITECTURE BASELINES (MEAN ± SD ACROSS SEEDS)")
print("=" * 100)
show = summary_df.copy()
show['Val AUC (mean±sd)'] = show.apply(lambda r: f"{r.Val_AUC_mean:.4f} ± {r.Val_AUC_std:.4f}", axis=1)
show['Test AUC (mean±sd)'] = show.apply(lambda r: f"{r.Test_AUC_mean:.4f} ± {r.Test_AUC_std:.4f}", axis=1)
show['Test F1 (mean±sd)'] = show.apply(lambda r: f"{r.Test_F1_mean:.4f} ± {r.Test_F1_std:.4f}", axis=1)
show = show[['Model','Seeds','Val AUC (mean±sd)','Test AUC (mean±sd)','Test F1 (mean±sd)','Parameters','Checkpoint_MB']]
print(show.to_string(index=False))
show.to_csv(os.path.join(OUT_DIR, 'cods_paper_architecture_table.csv'), index=False)
print(f"\n✓ Saved: {OUT_DIR}/cods_paper_architecture_table.csv")

print("\nRECOMMENDED REPORTING RULES")
print("  • Architecture comparison: mean ± SD over seeds.")
print("  • Deployment optimization: report FP32/FP16/INT8 with the fixed golden MobileViT checkpoint.")
print("  • Never select a final model using test AUC.")
print("  • State train subset/epochs explicitly; use full PCam training data for the archival run.")

---
## NB6 — Final Analysis & Pareto Visualisation
Generate paper-quality tables and three Pareto-frontier plots.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

matplotlib.rcParams.update({
    'font.family':   'DejaVu Sans',
    'font.size':     11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.alpha':         0.3,
    'figure.dpi':         150,
})

# Filter to GPU-inference rows for latency plots (or CPU if no GPU)
df_plot = df_results[df_results['Device'].isin(['T4 GPU', 'CPU'])].copy()

COLORS = {'FP32': '#2196F3', 'FP16': '#4CAF50', 'INT8': '#FF5722'}
MARKERS = {'T4 GPU': 'o', 'CPU': 's'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: AUC vs Latency ───────────────────────────────────────────────────
ax = axes[0]
for _, row in df_plot.dropna(subset=['Median_ms']).iterrows():
    ax.scatter(row['Median_ms'], row['Test_AUC'],
               color=COLORS.get(row['Precision'], 'grey'),
               marker=MARKERS.get(row['Device'], 'o'),
               s=120, zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(row['Variant'], (row['Median_ms'], row['Test_AUC']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Median Latency (ms) @ batch=1')
ax.set_ylabel('Test AUC')
ax.set_title('AUC vs Latency')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

# ── Plot 2: AUC vs Model Size ────────────────────────────────────────────────
ax = axes[1]
for _, row in df_plot.iterrows():
    ax.scatter(row['Size_MB'], row['Test_AUC'],
               color=COLORS.get(row['Precision'], 'grey'),
               marker=MARKERS.get(row['Device'], 'o'),
               s=120, zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(row['Variant'], (row['Size_MB'], row['Test_AUC']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Model Size (MB)')
ax.set_ylabel('Test AUC')
ax.set_title('AUC vs Model Size')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

# ── Plot 3: AUC Retention vs Speedup ────────────────────────────────────────
# BUGFIX: previously plotted against Speedup_x, a single GPU-FP32-relative
# column that made every CPU row (INT8, FP32 CPU) look far slower than it
# really is relative to its own device. Speedup_x_same_device compares each
# point to the FP32 baseline on its own device, so the plot isolates the
# precision effect instead of mixing it with a device effect.
ax = axes[2]
for _, row in df_plot.dropna(subset=['Speedup_x_same_device']).iterrows():
    ax.scatter(row['Speedup_x_same_device'], row['AUC_Retention_pct'],
               color=COLORS.get(row['Precision'], 'grey'),
               marker=MARKERS.get(row['Device'], 'o'),
               s=120, zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(row['Variant'], (row['Speedup_x_same_device'], row['AUC_Retention_pct']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.axhline(100.0, color='grey', linestyle='--', linewidth=0.8, label='FP32 baseline (same device)')
ax.set_xlabel('Speedup (×) vs same-device FP32')
ax.set_ylabel('AUC Retention (%)')
ax.set_title('AUC Retention vs Speedup (device-matched)')
ax.legend(fontsize=8)

# Legend for precision
from matplotlib.lines import Line2D
legend_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                          markersize=9, label=p) for p, c in COLORS.items()]
fig.legend(handles=legend_handles, title='Precision', loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, -0.08), frameon=False)

fig.suptitle('MobileViT-S PCam: Accuracy–Efficiency Trade-off (Kaggle T4)', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'pareto_analysis.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(OUT_DIR, 'pareto_analysis.png'), bbox_inches='tight')
plt.show()
print(f'✓ Pareto plots saved to {OUT_DIR}/')

In [ ]:
# ── Paper-ready Tables ────────────────────────────────────────────────────────
print('=' * 80)
print('TABLE 1 — Model Accuracy')
print('=' * 80)
t1_cols = ['Variant', 'Precision', 'Test_AUC', 'AUC_CI_low', 'AUC_CI_high', 'AUC_Retention_pct', 'Delta_AUC',
           'F1_fixed_thresh_0.5', 'F1_calibrated', 'Calib_Threshold']
print(df_results[t1_cols].to_string(index=False))
print('\n(AUC_CI_low/high: 95% bootstrap CI on Test_AUC, 1000 resamples.)')

print()
print()
print('=' * 80)
print('TABLE 1b — Sensitivity / Specificity (Test set)')
print('=' * 80)
t1b_cols = ['Variant', 'Precision', 'Sensitivity_fixed_0.5', 'Specificity_fixed_0.5',
            'Sensitivity_calibrated', 'Specificity_calibrated']
print(df_results[t1b_cols].to_string(index=False))
print()
print('Sensitivity = recall on tumor-positive class (low = missed-tumor risk).')
print('Specificity = recall on normal-negative class (low = false-alarm risk).')

print('=' * 80)
print('TABLE 2 — Computational Efficiency')
print('=' * 80)
t2_cols = ['Variant', 'Precision', 'Device', 'Size_MB', 'Compression_x', 'Median_ms',
           'p95_ms', 'FPS', 'Speedup_x_same_device', 'Speedup_x_vs_GPU_FP32']
print(df_results[t2_cols].to_string(index=False))
print()
print('Speedup_x_same_device: precision effect only (matched device baseline).')
print('Speedup_x_vs_GPU_FP32: cross-device deployment picture (CPU rows will')
print('                       always look slow here regardless of precision).')

print()
print('=' * 80)
print('ARCHITECTURE REFERENCE')
print('=' * 80)
print(f'  Backbone:    {CONFIG["BACKBONE"]}')
print(f'  Input size:  {CONFIG["IMG_SIZE"]}x{CONFIG["IMG_SIZE"]} px')
print(f'  Parameters:  {total_params:,}')
print(f'  MACs:        {total_macs / 1e6:.1f} M')
print(f'  FP32 size:   {fp32_size_mb:.2f} MB')


In [ ]:
# ── Final summary & best operating point ─────────────────────────────────────
# BUGFIX: previously scored by AUC_Retention_pct * Speedup_x, where Speedup_x
# was the device-confounded column (bug #3). Because that metric made GPU
# look no faster than CPU (bug #2) and made INT8/CPU look artificially slow
# vs a GPU baseline it never ran on, the score spuriously favored the
# unoptimized FP32 (CPU) row. Scoring on Speedup_x_same_device removes both
# confounds so the ranking reflects real precision/device trade-offs.
best_auc_ret = df_results.dropna(subset=['Speedup_x_same_device']).copy()
best_auc_ret['score'] = best_auc_ret['AUC_Retention_pct'] * best_auc_ret['Speedup_x_same_device']
best_row = best_auc_ret.loc[best_auc_ret['score'].idxmax()]

print('\n' + '=' * 60)
print('OPTIMAL OPERATING POINT')
print('=' * 60)
print(f'  Variant:         {best_row["Variant"]}')
print(f'  Test AUC:        {best_row["Test_AUC"]:.4f}')
print(f'  AUC Retention:   {best_row["AUC_Retention_pct"]:.2f}%')
print(f'  F1 (calibrated): {best_row["F1_calibrated"]:.4f}  (fixed-0.5: {best_row["F1_fixed_thresh_0.5"]:.4f})')
print(f'  Speedup (same-device baseline): {best_row["Speedup_x_same_device"]}x')
print(f'  Size:            {best_row["Size_MB"]} MB ({best_row["Compression_x"]}x smaller)')
print(f'  Median Latency:  {best_row["Median_ms"]} ms @ batch=1')
print()
print('NOTE: All measurements on Kaggle NVIDIA T4 GPU (GPU variants) and')
print('      Kaggle CPU (INT8 variant). Reported as resource-constrained')
print('      computational efficiency — not direct edge-device deployment.')
print('      GPU latency is IOBinding-based compute-only timing (bug #2 fix);')
print('      speedup is scored against a same-device FP32 baseline (bug #3 fix).')


## Cross-Architecture Pareto Frontier

Every Pareto/scatter plot so far only compares MobileViT-S against **itself** at different precisions (FP32 vs FP16 vs INT8). That answers "does quantization hurt MobileViT?" but not the more important reviewer question from the architecture-baseline study: **is MobileViT-S actually competitive against the newer mobile/edge architectures on accuracy vs. efficiency, or is a plain CNN just as good?**

This section benchmarks every architecture baseline (best-seed checkpoint, native PyTorch, FP32, CPU, batch=1 — same device/precision for everyone, so there's no device or precision confound) and plots them against MobileViT-S on the same axes. MobileViT-S's own FP16/INT8 deployment-optimized points are overlaid separately so you can see both (a) how MobileViT-S ranks among peer architectures at native FP32, and (b) how far precision optimization can move it beyond that.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-architecture Pareto frontier: benchmark every baseline's best-seed
# checkpoint (native PyTorch, FP32, CPU, batch=1) and compare against
# MobileViT-S on the SAME device/precision -- avoids the device/precision
# confound that the notebook already fixes elsewhere for the ONNX variants.
# ─────────────────────────────────────────────────────────────────────────────

def benchmark_torch_cpu(model, input_shape=(1, 3, 96, 96), warmup=50, iters=200):
    """Batch=1 CPU latency for a plain PyTorch model (no ONNX export)."""
    model = model.to('cpu').eval()
    dummy = torch.randn(*input_shape)
    with torch.no_grad():
        for _ in range(warmup):
            model(dummy)
        times_ms = []
        for _ in range(iters):
            t0 = time.perf_counter()
            model(dummy)
            times_ms.append((time.perf_counter() - t0) * 1000.0)
    times_ms = np.asarray(times_ms)
    return {
        'median_ms': float(np.median(times_ms)),
        'p95_ms': float(np.percentile(times_ms, 95)),
        'throughput_fps': float(1000.0 / np.median(times_ms)),
    }

# One checkpoint per architecture: the best-val-AUC seed (never test-selected).
best_ckpt_per_model = (
    baseline_seed_df.loc[baseline_seed_df.groupby('Model')['Best_Val_AUC'].idxmax()]
    .set_index('Model')
)

arch_pareto_rows = []

print("Benchmarking architecture baselines on CPU (native FP32, batch=1)...")
for model_name in CONFIG['BASELINE_MODELS']:
    ckpt_path = best_ckpt_per_model.loc[model_name, 'Checkpoint']
    m = timm.create_model(model_name, pretrained=False, num_classes=2)
    m.load_state_dict(torch.load(ckpt_path, map_location='cpu'), strict=True)
    bench_r = benchmark_torch_cpu(m, input_shape=(1, 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']))
    del m

    summary_row = summary_df[summary_df['Model'] == model_name].iloc[0]
    arch_pareto_rows.append({
        'Architecture':   model_name,
        'Family':         'Baseline (post-2021 mobile/edge)' if model_name in
                           ('edgenext_x_small', 'mobilevitv2_050', 'fastvit_t8', 'repvit_m0_9')
                           else 'Baseline (pre/same-gen CNN)',
        'Test_AUC_mean':  round(float(summary_row['Test_AUC_mean']), 4),
        'Test_AUC_std':   round(float(summary_row['Test_AUC_std']), 4),
        'Parameters':     int(summary_row['Parameters']),
        'Size_MB':        round(float(summary_row['Checkpoint_MB']), 2),
        'Median_ms_CPU':  round(bench_r['median_ms'], 3),
        'p95_ms_CPU':     round(bench_r['p95_ms'], 3),
    })
    print(f"  {model_name:<24} {bench_r['median_ms']:>8.3f} ms median")

# MobileViT-S itself, same protocol (native FP32, CPU, batch=1), using the
# already-loaded golden deploy_model and its own seed-robustness summary.
mobilevit_bench_cpu = benchmark_torch_cpu(
    copy.deepcopy(deploy_model), input_shape=(1, 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'])
)
mobilevit_summary_row = summary_df[summary_df['Model'] == CONFIG['BACKBONE']].iloc[0]
arch_pareto_rows.append({
    'Architecture':   CONFIG['BACKBONE'],
    'Family':         'MobileViT-S (this paper, native FP32)',
    'Test_AUC_mean':  round(float(mobilevit_summary_row['Test_AUC_mean']), 4),
    'Test_AUC_std':   round(float(mobilevit_summary_row['Test_AUC_std']), 4),
    'Parameters':     int(mobilevit_summary_row['Parameters']),
    'Size_MB':        round(float(mobilevit_summary_row['Checkpoint_MB']), 2),
    'Median_ms_CPU':  round(mobilevit_bench_cpu['median_ms'], 3),
    'p95_ms_CPU':     round(mobilevit_bench_cpu['p95_ms'], 3),
})
print(f"  {CONFIG['BACKBONE']:<24} {mobilevit_bench_cpu['median_ms']:>8.3f} ms median  (native FP32 reference)")

arch_pareto_df = pd.DataFrame(arch_pareto_rows)
arch_pareto_df.to_csv(os.path.join(OUT_DIR, 'cross_architecture_pareto_table.csv'), index=False)

print("\n" + "=" * 100)
print("TABLE — CROSS-ARCHITECTURE COMPARISON (native FP32, CPU, batch=1)")
print("=" * 100)
print(arch_pareto_df.to_string(index=False))
print(f"\n\u2713 Saved to {OUT_DIR}/cross_architecture_pareto_table.csv")


### Pareto plot

In [ ]:
# ── Plot: cross-architecture Pareto frontier ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

FAMILY_COLORS = {
    'Baseline (pre/same-gen CNN)':          '#9E9E9E',
    'Baseline (post-2021 mobile/edge)':     '#2196F3',
    'MobileViT-S (this paper, native FP32)':'#E91E63',
}

# Panel 1: AUC (mean ± cross-seed SD) vs CPU latency -- the headline plot.
ax = axes[0]
for _, row in arch_pareto_df.iterrows():
    color = FAMILY_COLORS.get(row['Family'], 'grey')
    is_mvit = row['Architecture'] == CONFIG['BACKBONE']
    ax.errorbar(
        row['Median_ms_CPU'], row['Test_AUC_mean'], yerr=row['Test_AUC_std'],
        fmt='*' if is_mvit else 'o', color=color,
        markersize=16 if is_mvit else 10, capsize=3, zorder=5,
        markeredgecolor='white', markeredgewidth=0.8,
    )
    ax.annotate(row['Architecture'], (row['Median_ms_CPU'], row['Test_AUC_mean']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Median CPU Latency (ms) @ batch=1, native FP32')
ax.set_ylabel('Test AUC (mean \u00b1 cross-seed SD)')
ax.set_title('Accuracy vs Latency — Architecture Comparison')

# Overlay MobileViT-S's own deployment-optimized points (FP16 GPU / INT8 CPU)
# from df_results, so the plot also shows what precision optimization buys
# on top of the architecture's own native-FP32 position (shown as the star).
mvit_deploy_rows = df_results[df_results['Variant'].str.startswith('MobileViT')]
for _, row in mvit_deploy_rows.iterrows():
    if row['Median_ms'] is None:
        continue
    ax.scatter(row['Median_ms'], row['Test_AUC'], marker='^', s=90,
                color='#E91E63', alpha=0.45, zorder=4,
                edgecolors='white', linewidths=0.6)
    ax.annotate(row['Variant'].replace('MobileViT ', ''),
                (row['Median_ms'], row['Test_AUC']),
                textcoords='offset points', xytext=(6, -10), fontsize=7, alpha=0.7)

# Panel 2: AUC vs Parameters -- device-independent efficiency view.
ax = axes[1]
for _, row in arch_pareto_df.iterrows():
    color = FAMILY_COLORS.get(row['Family'], 'grey')
    is_mvit = row['Architecture'] == CONFIG['BACKBONE']
    ax.errorbar(
        row['Parameters'] / 1e6, row['Test_AUC_mean'], yerr=row['Test_AUC_std'],
        fmt='*' if is_mvit else 'o', color=color,
        markersize=16 if is_mvit else 10, capsize=3, zorder=5,
        markeredgecolor='white', markeredgewidth=0.8,
    )
    ax.annotate(row['Architecture'], (row['Parameters'] / 1e6, row['Test_AUC_mean']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Test AUC (mean \u00b1 cross-seed SD)')
ax.set_title('Accuracy vs Model Size — Architecture Comparison')

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=9, label=f)
    for f, c in FAMILY_COLORS.items()
]
legend_handles.append(
    Line2D([0],[0], marker='^', color='w', markerfacecolor='#E91E63', alpha=0.45,
           markersize=9, label='MobileViT-S deployment-optimized (FP16/INT8)')
)
fig.legend(handles=legend_handles, loc='lower center', ncol=1, fontsize=8,
           bbox_to_anchor=(0.5, -0.22), frameon=False)

fig.suptitle('Cross-Architecture Accuracy–Efficiency Comparison (PCam, native FP32 unless noted)',
             fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'cross_architecture_pareto.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(OUT_DIR, 'cross_architecture_pareto.png'), bbox_inches='tight')
plt.show()
print(f"\u2713 Cross-architecture Pareto plots saved to {OUT_DIR}/")


## Conservative INT8 CPU Benchmark


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CPU Benchmark — deployment-oriented, batch=1
#
# The previous results showed poor INT8 scaling at high thread counts.
# Use 2 intra-op threads / 1 inter-op thread as the default comparison point.
# ─────────────────────────────────────────────────────────────────────────────

def benchmark_onnx_cpu(
    onnx_path,
    input_np_fp32,
    warmup=100,
    iters=500,
    intra_threads=2,
):
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.intra_op_num_threads = intra_threads
    sess_opts.inter_op_num_threads = 1
    sess_opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    sess_opts.log_severity_level = 3

    sess = ort.InferenceSession(
        onnx_path,
        sess_options=sess_opts,
        providers=["CPUExecutionProvider"],
    )

    input_name = sess.get_inputs()[0].name

    for _ in range(warmup):
        sess.run(None, {input_name: input_np_fp32})

    times_ms = []
    for _ in range(iters):
        t0 = time.perf_counter()
        sess.run(None, {input_name: input_np_fp32})
        times_ms.append((time.perf_counter() - t0) * 1000.0)

    times_ms = np.asarray(times_ms)

    return {
        "median_ms": float(np.median(times_ms)),
        "p95_ms": float(np.percentile(times_ms, 95)),
        "p99_ms": float(np.percentile(times_ms, 99)),
        "throughput_fps": float(1000.0 / np.median(times_ms)),
        "all_times_ms": times_ms,
    }


print("ORT version:", ort.__version__)
print("Available providers:", ort.get_available_providers())
print()

bench_fp32_cpu = benchmark_onnx_cpu(
    ONNX_FP32,
    bench_img_np,
    warmup=100,
    iters=500,
    intra_threads=2,
)

bench_int8_cpu = benchmark_onnx_cpu(
    ONNX_INT8,
    bench_img_np,
    warmup=100,
    iters=500,
    intra_threads=2,
)

print("=" * 80)
print("CPU LATENCY")
print("=" * 80)
print(
    f"FP32 : {bench_fp32_cpu['median_ms']:.3f} ms median | "
    f"{bench_fp32_cpu['p95_ms']:.3f} ms p95 | "
    f"{bench_fp32_cpu['throughput_fps']:.1f} FPS"
)
print(
    f"INT8 : {bench_int8_cpu['median_ms']:.3f} ms median | "
    f"{bench_int8_cpu['p95_ms']:.3f} ms p95 | "
    f"{bench_int8_cpu['throughput_fps']:.1f} FPS"
)
print(
    f"INT8 / FP32 latency ratio: "
    f"{bench_int8_cpu['median_ms'] / bench_fp32_cpu['median_ms']:.2f}x"
)


## Precision Ablation — Full Candidate Comparison

The mixed-precision INT8 selection above picks a winner (`selected_candidate`) using validation AUC only, then re-quantizes just that winner with the full calibration set — which is correct methodology, but it also means the *losing* candidate's numbers never get reported anywhere. A reviewer asking "compared with what?" about the mixed-precision strategy itself needs to see **both** candidates side by side, not just the one that won.

This table reports, for every INT8 Conv-quantization candidate plus the final re-quantized selection and an FP32 CPU reference:
- how many Conv nodes were quantized,
- how many calibration images were used (the two screening candidates used a smaller `SCREEN_CALIB_N`; the final selected model was re-quantized with the full `CONFIG['CALIB_IMAGES']` set — these are **not** apples-to-apples on calibration size, which the table makes explicit),
- validation AUC (the quantity actually used for selection),
- test AUC with a 95% bootstrap CI (reported for completeness; **not** used for selection),
- CPU latency and model size.

Test-set numbers here are diagnostic/reporting only — the selection decision earlier in this notebook was and remains validation-only.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Full precision-ablation table: report ALL INT8 candidates, not just the winner
#
# candidate_paths / candidate_specs / candidate_val_scores / selected_candidate
# were all set earlier during candidate selection. Selection there used
# validation AUC only; nothing below changes that decision -- this cell adds
# test-set numbers, latency, and size for every candidate purely for
# transparent ablation reporting.
# ─────────────────────────────────────────────────────────────────────────────

ablation_rows = []

for candidate_name, candidate_path in candidate_paths.items():
    val_r  = evaluate_onnx(candidate_path, val_loader,  use_gpu=False, desc=f"Ablation {candidate_name} val")
    test_r = evaluate_onnx(candidate_path, test_loader, use_gpu=False, desc=f"Ablation {candidate_name} test")
    bench_r = benchmark_onnx_cpu(candidate_path, bench_img_np, warmup=100, iters=500, intra_threads=2)
    _, ci_lo, ci_hi = bootstrap_auc_ci(test_r['labels'], test_r['probs'], n_boot=1000, seed=0)

    ablation_rows.append({
        'Candidate':        candidate_name,
        'Selected':         (candidate_name == selected_candidate),
        'Conv_Nodes_Quantized': len(candidate_specs[candidate_name]),
        'Calib_Images':     SCREEN_CALIB_N,
        'Size_MB':          round(os.path.getsize(candidate_path) / (1024 ** 2), 2),
        'Val_AUC':          round(val_r['auc'], 4),
        'Test_AUC':         round(test_r['auc'], 4),
        'Test_AUC_CI_low':  round(ci_lo, 4) if not np.isnan(ci_lo) else None,
        'Test_AUC_CI_high': round(ci_hi, 4) if not np.isnan(ci_hi) else None,
        'Test_F1':          round(test_r['f1'], 4),
        'Median_ms':        round(bench_r['median_ms'], 3),
        'p95_ms':           round(bench_r['p95_ms'], 3),
    })

# Final re-quantized selection (full calibration set) -- reuses the accuracy
# numbers already computed earlier (onnx_int8_val / onnx_int8_test) and the
# latency already computed just above in this section (bench_int8_cpu), so
# it is not re-run.
_, final_ci_lo, final_ci_hi = bootstrap_auc_ci(
    onnx_int8_test['labels'], onnx_int8_test['probs'], n_boot=1000, seed=0
)
ablation_rows.append({
    'Candidate':        f'{selected_candidate} (FINAL, full calib)',
    'Selected':         True,
    'Conv_Nodes_Quantized': len(candidate_specs[selected_candidate]),
    'Calib_Images':     len(calib_indices),
    'Size_MB':          round(int8_size_mb, 2),
    'Val_AUC':          round(onnx_int8_val['auc'], 4),
    'Test_AUC':         round(onnx_int8_test['auc'], 4),
    'Test_AUC_CI_low':  round(final_ci_lo, 4) if not np.isnan(final_ci_lo) else None,
    'Test_AUC_CI_high': round(final_ci_hi, 4) if not np.isnan(final_ci_hi) else None,
    'Test_F1':          round(onnx_int8_test['f1'], 4),
    'Median_ms':        round(bench_int8_cpu['median_ms'], 3),
    'p95_ms':           round(bench_int8_cpu['p95_ms'], 3),
})

# FP32 CPU reference row for scale (reuses bench_fp32_cpu computed just above).
ablation_rows.append({
    'Candidate':        'FP32 (no quantization, CPU reference)',
    'Selected':         False,
    'Conv_Nodes_Quantized': 0,
    'Calib_Images':     None,
    'Size_MB':          round(fp32_size_mb, 2),
    'Val_AUC':          round(onnx_fp32_val['auc'], 4),
    'Test_AUC':         round(onnx_fp32_test['auc'], 4),
    'Test_AUC_CI_low':  None,
    'Test_AUC_CI_high': None,
    'Test_F1':          round(onnx_fp32_test['f1'], 4),
    'Median_ms':        round(bench_fp32_cpu['median_ms'], 3),
    'p95_ms':           round(bench_fp32_cpu['p95_ms'], 3),
})

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(os.path.join(OUT_DIR, 'mixed_precision_ablation_table.csv'), index=False)

print("=" * 100)
print("TABLE — MIXED-PRECISION INT8 CANDIDATE ABLATION")
print("=" * 100)
print(ablation_df.to_string(index=False))
print(f"\n\u2713 Saved to {OUT_DIR}/mixed_precision_ablation_table.csv")
print("\nNOTE: the two screening candidates (SAFE_CONV, STEM_CONV) were quantized with")
print("      only SCREEN_CALIB_N calibration images for fast comparison; the FINAL row")
print("      re-quantizes the SELECTED candidate with the full calibration set -- sizes")
print("      and AUCs are not directly comparable to the screening rows on that axis.")
print("NOTE: selection used Val_AUC only, decided earlier in the notebook. Test_AUC and")
print("      its bootstrap CI are reported here for transparency, not for selection.")


## INT8 Graph Sanity Check


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECOND-GEN INT8 GRAPH + EXECUTION-PROVIDER SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────

import onnx
from collections import Counter

model = onnx.load(ONNX_INT8)
op_counts = Counter(node.op_type for node in model.graph.node)

print("=" * 80)
print("SECOND-GENERATION MIXED-PRECISION INT8 GRAPH")
print("=" * 80)
print("Selected candidate:", selected_candidate)
print("Quantized Conv nodes:", len(selected_nodes))

for op, count in op_counts.most_common():
    print(f"{op:<30} {count}")

print("\nQuantization operators:")
for op in [
    "QuantizeLinear", "DequantizeLinear", "QLinearConv", "ConvInteger",
    "MatMulInteger", "QLinearMatMul"
]:
    print(f"{op:<20}: {op_counts.get(op, 0)}")

# Verify the final graph can initialize on the intended CPU EP.
try:
    cpu_opts = ort.SessionOptions()
    cpu_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    cpu_opts.log_severity_level = 3
    cpu_sess = ort.InferenceSession(
        ONNX_INT8,
        sess_options=cpu_opts,
        providers=["CPUExecutionProvider"],
    )
    print("\n✓ CPUExecutionProvider session initialized")
    print("  Providers:", cpu_sess.get_providers())
    print("  Input:", cpu_sess.get_inputs()[0].name, cpu_sess.get_inputs()[0].shape)
except Exception as e:
    print("\n✗ CPUExecutionProvider initialization failed:", type(e).__name__, e)

print("\nNOTE: QDQ nodes are expected in a QDQ-format graph. Their presence alone")
print("does NOT prove that execution is FP32; the actual execution provider/kernel")
print("selection determines whether quantized Conv kernels are used.")


## Second-Generation Mixed-Precision INT8 Evaluation


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Accuracy Evaluation + Validation-Only Threshold Calibration
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 80)
print("SECOND-GENERATION MIXED-PRECISION INT8 ACCURACY")
print("=" * 80)

onnx_int8_val = evaluate_onnx(
    ONNX_INT8,
    val_loader,
    use_gpu=False,
    desc="Mixed INT8 Val",
)

onnx_int8_test = evaluate_onnx(
    ONNX_INT8,
    test_loader,
    use_gpu=False,
    desc="Mixed INT8 Test",
)

print("\nRaw metrics:")
print(f"Val  AUC: {onnx_int8_val['auc']:.4f}")
print(f"Test AUC: {onnx_int8_test['auc']:.4f}")

# Calibrate threshold on validation only.
thresh, val_f1_at_thresh = best_f1_threshold(
    onnx_int8_val["probs"],
    onnx_int8_val["labels"],
)

test_preds_cal = (
    onnx_int8_test["probs"] >= thresh
).astype(int)

test_f1_cal = f1_score(
    onnx_int8_test["labels"],
    test_preds_cal,
)

test_sens_cal, test_spec_cal = sens_spec(
    onnx_int8_test["labels"], onnx_int8_test["probs"], thresh
)



print("\nCalibrated metrics:")
print(f"Threshold       : {thresh:.6f}")
print(f"Val F1          : {val_f1_at_thresh:.4f}")
print(f"Test F1         : {test_f1_cal:.4f}")
print(f"Test sensitivity: {test_sens_cal:.4f}")
print(f"Test specificity: {test_spec_cal:.4f}")

if "auc" in onnx_int8_test:
    auc_ret = onnx_int8_test["auc"] / fp32_test_auc * 100 if "fp32_test_auc" in globals() else None
    if auc_ret is not None:
        print(f"AUC retention    : {auc_ret:.2f}%")


## Final Deployment Summary


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Final Deployment Summary — Second-Generation Mixed Precision
# ─────────────────────────────────────────────────────────────────────────────

size_mb = os.path.getsize(ONNX_INT8) / (1024 ** 2)
compression = fp32_size_mb / size_mb
lat_ratio = bench_int8_cpu["median_ms"] / bench_fp32_cpu["median_ms"]

print("=" * 80)
print("DEPLOYMENT SUMMARY — MIXED-PRECISION INT8")
print("=" * 80)
print(f"Model size       : {size_mb:.2f} MB")
print(f"Compression      : {compression:.2f}x")
print(f"FP32 CPU median  : {bench_fp32_cpu['median_ms']:.3f} ms")
print(f"INT8 CPU median  : {bench_int8_cpu['median_ms']:.3f} ms")
print(f"Latency ratio    : {lat_ratio:.2f}x")
print(f"Test AUC         : {onnx_int8_test['auc']:.4f}")
print(f"Calibrated F1    : {test_f1_cal:.4f}")
print(f"Calibrated sens. : {test_sens_cal:.4f}")

print("\nInterpretation:")
if onnx_int8_test["auc"] >= 0.94:
    print("✓ Accuracy is substantially closer to FP32; mixed-precision INT8 is a strong candidate.")
elif onnx_int8_test["auc"] >= 0.92:
    print("△ Accuracy is improved but still below the FP32 target; review the deployment requirement.")
else:
    print("✗ PTQ accuracy remains materially degraded; the next step is QAT or a more selective FP32/INT8 partition.")

if lat_ratio < 1.0:
    print("✓ INT8 is faster than FP32 in this CPU configuration.")
else:
    print("△ INT8 is not faster than FP32 in this CPU configuration; backend/kernel optimization remains a separate issue.")


## Fourth-Generation: Conv+MatMul Mixed Precision — Closing the INT8 Latency Gap

The second-generation INT8 above (`ONNX_INT8`, Step C2) deliberately keeps `MatMul` in
`SENSITIVE_OPS` and only quantizes a subset of `Conv` nodes. That was the right call for
accuracy, but it has a latency consequence worth making explicit: this backbone has 35 `Conv`
nodes and 54 `MatMul` nodes (the attention/FFN layers), so a Conv-only quantization leaves the
majority of the model's compute in FP32 while still paying the QuantizeLinear/DequantizeLinear
conversion cost at every quantized boundary. That's the most likely reason the Conservative INT8
CPU Benchmark section above shows INT8 at or slower than FP32 rather than faster.

This section runs a second axis of the same validation-only screening methodology already used
for `SAFE_CONV` / `STEM_CONV`, extended to `MatMul`:

1. Classify `MatMul` nodes by graph proximity to `Softmax` (2-hop BFS) — nodes near a Softmax are
   the attention-score matmuls (`QK^T` and `softmax(QK^T) @ V`), which are the ones a ViT
   quantization literature review would flag as precision-sensitive. Nodes further away are
   mostly FFN/projection linear layers, which tend to quantize much more safely.
2. Quantize `Conv` (existing `safe_conv_nodes`) **and** the "safe" `MatMul` set together, and
   compare **QDQ format** (matches the existing pipeline's format, isolates the op-coverage
   effect) against **QOperator format** (fuses into `QLinearConv`/`QLinearMatMul` kernels directly
   — isolates the format effect on CPU latency).
3. Selection is validation-AUC-only, same rule as Step C2, then the winner is re-quantized with
   the full calibration set and benchmarked with the same thread-matched harness (`intra_threads=2`)
   used in the Conservative INT8 CPU Benchmark section, so the comparison to `bench_fp32_cpu` is
   apples-to-apples.


In [ ]:
# ── Step D1: Classify MatMul nodes by proximity to Softmax ──────────────────
# Same spirit as the SAFE_CONV / STEM_CONV split in Step C2, extended to MatMul.
# MatMul nodes within 2 hops of a Softmax node are treated as attention-score-
# sensitive and left FP32; the rest (mostly FFN/projection linear layers) are
# quantization candidates.

from collections import defaultdict, Counter

gen4_model = onnx.load(ONNX_FP32_PREP)
gen4_nodes = list(gen4_model.graph.node)

gen4_producer_of = {}
for n in gen4_nodes:
    for out in n.output:
        gen4_producer_of[out] = n

gen4_consumers = defaultdict(list)
for n in gen4_nodes:
    for inp in n.input:
        gen4_consumers[inp].append(n)

def _gen4_neighbors(n):
    out = []
    for inp in n.input:
        p = gen4_producer_of.get(inp)
        if p is not None:
            out.append(p)
    for o in n.output:
        out.extend(gen4_consumers.get(o, []))
    return out

softmax_nodes = [n for n in gen4_nodes if n.op_type == "Softmax"]
sensitive_ids = set()
frontier = list(softmax_nodes)
visited = set(id(n) for n in frontier)
for _hop in range(2):  # 2-hop BFS from every Softmax node
    nxt = []
    for n in frontier:
        for nb in _gen4_neighbors(n):
            if id(nb) not in visited:
                visited.add(id(nb))
                nxt.append(nb)
    frontier = nxt
    sensitive_ids.update(id(n) for n in frontier if n.op_type == "MatMul")

matmul_nodes = [n for n in gen4_nodes if n.op_type == "MatMul"]
safe_matmul_nodes = [n.name for n in matmul_nodes if id(n) not in sensitive_ids]
sensitive_matmul_nodes = [n.name for n in matmul_nodes if id(n) in sensitive_ids]

print(f"Total MatMul nodes      : {len(matmul_nodes)}")
print(f"Sensitive (near Softmax): {len(sensitive_matmul_nodes)}  (kept FP32)")
print(f"Safe (FFN/projection)   : {len(safe_matmul_nodes)}  (quantization candidates)")

if "safe_conv_nodes" not in globals():
    raise RuntimeError(
        "safe_conv_nodes is not defined -- run the Step C2 mixed-precision cell first "
        "(this section builds on its Conv classification)."
    )
print(f"\nReusing safe_conv_nodes from Step C2: {len(safe_conv_nodes)} Conv nodes")


In [ ]:
# ── Step D2: Quantize Conv+MatMul candidates (screen calibration set) ───────
# Two candidates, same node set (safe_conv_nodes + safe_matmul_nodes), to
# separate the op-coverage effect from the format effect:
#   CONV_MATMUL_QDQ       -- QDQ format (matches Step C2's format exactly)
#   CONV_MATMUL_QOPERATOR -- QOperator format (fuses to QLinearConv/QLinearMatMul,
#                             the change actually expected to move CPU latency)

gen4_candidate_dir = os.path.join(OUT_DIR, "gen4_conv_matmul_candidates")
os.makedirs(gen4_candidate_dir, exist_ok=True)

gen4_node_set = list(safe_conv_nodes) + list(safe_matmul_nodes)

def quantize_nodes_v2(node_names, output_path, calibration_indices, op_types,
                       quant_format, method=CalibrationMethod.MinMax):
    '''Like Step C2's quantize_nodes(), generalized to arbitrary op_types_to_quantize
    and quant_format so it can cover Conv+MatMul and both QDQ/QOperator without
    touching the original helper (which downstream cells 29/49/53/55 still depend on).'''
    if os.path.exists(output_path):
        os.remove(output_path)
    reader = make_calib_reader(calibration_indices)
    quantize_static(
        model_input=ONNX_FP32_PREP,
        model_output=output_path,
        calibration_data_reader=reader,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
        quant_format=quant_format,
        per_channel=False,
        calibrate_method=method,
        nodes_to_quantize=list(node_names),
        op_types_to_quantize=op_types,
    )
    if not os.path.exists(output_path):
        raise RuntimeError(f"Quantization failed to create {output_path}")

gen4_candidate_specs = {
    "CONV_MATMUL_QDQ":       (gen4_node_set, QuantFormat.QDQ),
    "CONV_MATMUL_QOPERATOR": (gen4_node_set, QuantFormat.QOperator),
}

gen4_candidate_paths = {}
for cand_name, (node_names, fmt) in gen4_candidate_specs.items():
    path = os.path.join(gen4_candidate_dir, f"mobilevit_{cand_name.lower()}_screen.onnx")
    print(f"\nQuantizing {cand_name}: {len(node_names)} nodes "
          f"({len(safe_conv_nodes)} Conv + {len(safe_matmul_nodes)} MatMul), format={fmt}")
    t0 = time.time()
    quantize_nodes_v2(node_names, path, screen_indices, op_types=["Conv", "MatMul"], quant_format=fmt)
    gen4_candidate_paths[cand_name] = path
    print(f"  \u2713 {cand_name}: {time.time() - t0:.1f}s | {os.path.getsize(path)/(1024**2):.2f} MB")


In [ ]:
# ── Step D3: Validation-only selection + full re-quantization of the winner ─
# Same rule as Step C2: select on validation AUC only, never on test.

print("=" * 80)
print("GEN-4 (CONV+MATMUL) CANDIDATE SELECTION -- VALIDATION ONLY")
print("=" * 80)

gen4_val_scores = {}
for cand_name, cand_path in gen4_candidate_paths.items():
    try:
        r = evaluate_onnx(cand_path, val_loader, use_gpu=False, desc=f"Val {cand_name}")
        gen4_val_scores[cand_name] = r["auc"]
        print(f"{cand_name:<22} validation AUC: {r['auc']:.4f}")
    except Exception as e:
        gen4_val_scores[cand_name] = -np.inf
        print(f"{cand_name:<22} FAILED: {type(e).__name__}: {e}")

# Also compare against the existing Step C2 first-generation INT8 (Conv-only),
# already evaluated earlier as onnx_int8_val.
print(f"{'CONV_ONLY (Step C2)':<22} validation AUC: {onnx_int8_val['auc']:.4f}  (reference, already selected)")

if not any(np.isfinite(v) for v in gen4_val_scores.values()):
    raise RuntimeError("All Gen-4 Conv+MatMul candidates failed validation.")

gen4_selected = max(gen4_val_scores, key=gen4_val_scores.get)
gen4_selected_nodes, gen4_selected_fmt = gen4_candidate_specs[gen4_selected]
print(f"\n\u2713 Selected Gen-4 candidate: {gen4_selected}")
print(f"  Screening validation AUC: {gen4_val_scores[gen4_selected]:.4f}")

delta_vs_conv_only = gen4_val_scores[gen4_selected] - onnx_int8_val["auc"]
print(f"  Delta vs. Step C2 Conv-only (val AUC): {delta_vs_conv_only:+.4f}")

# Re-quantize the winner with the FULL calibration set, same as Step C2.
ONNX_INT8_GEN4 = os.path.join(OUT_DIR, "mobilevit_int8_conv_matmul.onnx")
print(f"\nRe-quantizing {gen4_selected} with {len(calib_indices)} calibration images...")
t0 = time.time()
quantize_nodes_v2(gen4_selected_nodes, ONNX_INT8_GEN4, calib_indices,
                   op_types=["Conv", "MatMul"], quant_format=gen4_selected_fmt)
print(f"\u2713 Final Gen-4 INT8 created in {time.time() - t0:.1f}s")

gen4_size_mb = os.path.getsize(ONNX_INT8_GEN4) / (1024 ** 2)
print(f"  Path: {ONNX_INT8_GEN4}")
print(f"  Size: {gen4_size_mb:.2f} MB  (Step C2 Conv-only was {int8_size_mb:.2f} MB)")


In [ ]:
# ── Step D4: Benchmark Gen-4 INT8 (thread-matched CPU) + test accuracy ──────
# Reuses benchmark_onnx_cpu(intra_threads=2) from the Conservative INT8 CPU
# Benchmark section above, so this is directly comparable to bench_fp32_cpu
# and bench_int8_cpu (Step C2, Conv-only) computed earlier in the notebook.

gen4_bench_cpu = benchmark_onnx_cpu(ONNX_INT8_GEN4, bench_img_np, warmup=100, iters=500, intra_threads=2)

gen4_val  = evaluate_onnx(ONNX_INT8_GEN4, val_loader,  use_gpu=False, desc="Gen4 INT8 Val")
gen4_test = evaluate_onnx(ONNX_INT8_GEN4, test_loader, use_gpu=False, desc="Gen4 INT8 Test")
_, gen4_ci_lo, gen4_ci_hi = bootstrap_auc_ci(gen4_test["labels"], gen4_test["probs"], n_boot=1000, seed=0)

gen4_lat_ratio_vs_fp32 = gen4_bench_cpu["median_ms"] / bench_fp32_cpu["median_ms"]
gen4_lat_ratio_vs_conv_only = gen4_bench_cpu["median_ms"] / bench_int8_cpu["median_ms"]

print("=" * 80)
print(f"GEN-4 INT8 (Conv+MatMul, {gen4_selected}) -- FINAL RESULT")
print("=" * 80)
print(f"Format             : {gen4_selected_fmt}")
print(f"Nodes quantized    : {len(gen4_selected_nodes)} "
      f"({len(safe_conv_nodes)} Conv + {len(safe_matmul_nodes)} MatMul)")
print(f"Size               : {gen4_size_mb:.2f} MB  (compression {fp32_size_mb/gen4_size_mb:.2f}x)")
print(f"Test AUC           : {gen4_test['auc']:.4f}  (95% CI [{gen4_ci_lo:.4f}, {gen4_ci_hi:.4f}])")
print(f"Test AUC (FP32 ref): {onnx_fp32_test['auc']:.4f}")
print(f"Test AUC (Step C2 Conv-only): {onnx_int8_test['auc']:.4f}")
print(f"CPU median latency : {gen4_bench_cpu['median_ms']:.3f} ms  "
      f"(FP32 CPU: {bench_fp32_cpu['median_ms']:.3f} ms, "
      f"Step C2 Conv-only INT8: {bench_int8_cpu['median_ms']:.3f} ms)")
print(f"Latency vs FP32 CPU        : {gen4_lat_ratio_vs_fp32:.2f}x "
      f"({'faster' if gen4_lat_ratio_vs_fp32 < 1.0 else 'slower'})")
print(f"Latency vs Step C2 Conv-only: {gen4_lat_ratio_vs_conv_only:.2f}x")

print("\nInterpretation:")
delta_test_auc = onnx_fp32_test["auc"] - gen4_test["auc"]
if delta_test_auc > 0.01:
    print(f"\u2717 Test AUC dropped {delta_test_auc:.4f} vs FP32 (> 0.01 threshold used elsewhere "
          f"in this notebook) -- the 'safe' MatMul heuristic (2-hop-from-Softmax) was too permissive "
          f"for at least one included layer. Consider narrowing to 1-hop, or dropping to per-layer "
          f"screening like SAFE_CONV/STEM_CONV.")
else:
    print(f"\u2713 Test AUC degradation acceptable: \u0394AUC = {delta_test_auc:.4f} (threshold 0.01)")

if gen4_lat_ratio_vs_fp32 < 1.0:
    print(f"\u2713 Gen-4 INT8 is faster than FP32 on CPU ({gen4_lat_ratio_vs_fp32:.2f}x) -- "
          f"quantizing MatMul closed the latency gap that Conv-only quantization left open.")
else:
    print(f"\u25b3 Gen-4 INT8 is still not faster than FP32 CPU ({gen4_lat_ratio_vs_fp32:.2f}x). "
          f"If CONV_MATMUL_QOPERATOR was selected and this is still true, the remaining gap is "
          f"likely CPU kernel/backend-level (check VNNI/AVX512-VNNI support, or try the "
          f"OpenVINO execution provider) rather than a quantization-coverage problem.")


In [ ]:
# ── Step D5: Consolidated comparison table + save CSV ────────────────────────
# FP32 CPU vs Step C2 (Conv-only) vs Gen-4 (Conv+MatMul) side by side.

gen4_rows = [
    {
        "Variant": "FP32 (CPU reference)",
        "Nodes_Quantized": 0,
        "Format": None,
        "Size_MB": round(fp32_size_mb, 2),
        "Test_AUC": round(onnx_fp32_test["auc"], 4),
        "Test_AUC_CI_low": None,
        "Test_AUC_CI_high": None,
        "Median_ms": round(bench_fp32_cpu["median_ms"], 3),
        "p95_ms": round(bench_fp32_cpu["p95_ms"], 3),
        "Speedup_vs_FP32": 1.0,
    },
    {
        "Variant": f"Step C2 INT8 (Conv-only, {selected_candidate})",
        "Nodes_Quantized": len(selected_nodes),
        "Format": "QDQ",
        "Size_MB": round(int8_size_mb, 2),
        "Test_AUC": round(onnx_int8_test["auc"], 4),
        "Test_AUC_CI_low": None,
        "Test_AUC_CI_high": None,
        "Median_ms": round(bench_int8_cpu["median_ms"], 3),
        "p95_ms": round(bench_int8_cpu["p95_ms"], 3),
        "Speedup_vs_FP32": round(bench_fp32_cpu["median_ms"] / bench_int8_cpu["median_ms"], 2),
    },
    {
        "Variant": f"Gen-4 INT8 (Conv+MatMul, {gen4_selected})",
        "Nodes_Quantized": len(gen4_selected_nodes),
        "Format": str(gen4_selected_fmt),
        "Size_MB": round(gen4_size_mb, 2),
        "Test_AUC": round(gen4_test["auc"], 4),
        "Test_AUC_CI_low": round(gen4_ci_lo, 4) if not np.isnan(gen4_ci_lo) else None,
        "Test_AUC_CI_high": round(gen4_ci_hi, 4) if not np.isnan(gen4_ci_hi) else None,
        "Median_ms": round(gen4_bench_cpu["median_ms"], 3),
        "p95_ms": round(gen4_bench_cpu["p95_ms"], 3),
        "Speedup_vs_FP32": round(bench_fp32_cpu["median_ms"] / gen4_bench_cpu["median_ms"], 2),
    },
]

gen4_df = pd.DataFrame(gen4_rows)
gen4_out_path = os.path.join(OUT_DIR, "mixed_precision_gen4_conv_matmul_table.csv")
gen4_df.to_csv(gen4_out_path, index=False)

print("=" * 100)
print("TABLE -- CONV-ONLY vs CONV+MATMUL MIXED PRECISION")
print("=" * 100)
print(gen4_df.to_string(index=False))
print(f"\n\u2713 Saved to {gen4_out_path}")
